# Engineering Intelligence Platform (EIP)

## 02. GitHub Historical Data Profiling

### 1. Objective

Evaluate historical GitHub pull request activity, source
data volumes and API consumption requirements to determine
an efficient initial ingestion strategy.

### 2. Source Information

| Attribute | Description |
|---|---|
| Source System | GitHub REST API |
| Repository | apache/airflow |
| Source Entity | Pull Requests |
| Source ID | SRC-002 |
| Authentication | Read-only personal access token |
| Time Standard | UTC |

### 3. Historical Profiling Windows

- Last 30 complete calendar days
- Last 60 complete calendar days
- Last 90 complete calendar days

The current incomplete UTC calendar day is excluded from
historical event counts.

### 4. Profiling Scope

- Historical pull request creation volume
- Historical pull request merge volume
- Historical pull request closure volume
- Pull request update activity
- Older pull requests with recent activity
- Weekly activity distribution
- Candidate ingestion volumes
- API request consumption estimates

### 5. Extraction Strategy

Pull requests are retrieved in descending order of their
latest update timestamp.

Pagination continues until the oldest retrieved update
timestamp precedes the historical cutoff, the source is
exhausted, or a configured request limit is reached.

Source records are deduplicated using stable GitHub IDs.

### 6. Expected Outcomes

- Recommended initial historical ingestion window
- Estimated Bronze ingestion volumes
- Estimated API request requirements
- Preliminary incremental ingestion strategy
- Historical coverage limitations

### 7. Dependencies

01_source_exploration/01_github_api_exploration.ipynb

This notebook independently initializes its environment
and does not require the previous notebook's kernel state.

### 8. Development Phase

Phase 02.03: Historical Data Profiling


In [1]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 01: Historical Profiling Configuration
#
# Purpose:
# Initialize dependencies, authenticate source access and
# establish configurable historical profiling windows.
# ============================================================


# ------------------------------------------------------------
# 01. Import dependencies
# ------------------------------------------------------------

import re
import sys
import time
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values
from IPython.display import display


# ------------------------------------------------------------
# 02. Identify project root
# ------------------------------------------------------------

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
        and (path / "backend").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Unable to locate EIP project root.")


# ------------------------------------------------------------
# 03. Load secure environment configuration
# ------------------------------------------------------------

ENV_FILE = PROJECT_ROOT / ".env"

if not ENV_FILE.is_file():
    raise FileNotFoundError("Local .env file is missing.")

config = dotenv_values(ENV_FILE)

GITHUB_TOKEN = config.get("GITHUB_TOKEN")
GITHUB_REPOSITORY = config.get("GITHUB_REPOSITORY")

if not GITHUB_TOKEN or not GITHUB_REPOSITORY:
    raise RuntimeError("Required GitHub configuration is missing.")

if not re.fullmatch(
    r"[A-Za-z0-9-]+/[A-Za-z0-9_.-]+",
    GITHUB_REPOSITORY,
):
    raise ValueError("Invalid repository identifier.")


# ------------------------------------------------------------
# 04. Configure GitHub API access
# ------------------------------------------------------------

GITHUB_BASE_URL = "https://api.github.com"
GITHUB_API_VERSION = "2026-03-10"

REQUEST_TIMEOUT_SECONDS = 30.0

REQUEST_HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": GITHUB_API_VERSION,
}


# ------------------------------------------------------------
# 05. Configure historical reporting windows
# ------------------------------------------------------------

# Use complete UTC calendar days to avoid including
# partial activity from the current day.
#
# Reporting windows follow:
# startDate <= eventTimestamp < endDate

PROFILE_WINDOWS_DAYS = [30, 60, 90]

AS_OF_TIMESTAMP = pd.Timestamp.now(tz="UTC").normalize()

HISTORICAL_CUTOFF = (
    AS_OF_TIMESTAMP
    - pd.Timedelta(days=max(PROFILE_WINDOWS_DAYS))
)


# ------------------------------------------------------------
# 06. Configure extraction safeguards
# ------------------------------------------------------------

# GitHub supports up to 100 PR records per list request.

PAGE_SIZE = 100

# Bound the exploration workload to 50 list requests.
# An incomplete scan will be explicitly identified.

MAX_PAGES = 50

# Maintain a reserve for subsequent development activities.

MIN_REMAINING_REQUESTS = 500

# Sequential requests with a short delay reduce API pressure.

REQUEST_DELAY_SECONDS = 0.25


# ------------------------------------------------------------
# 07. Configure Pandas display
# ------------------------------------------------------------

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)


# ------------------------------------------------------------
# 08. Display profiling configuration
# ------------------------------------------------------------

print("EIP | Historical Data Profiling")
print("=" * 65)

print("Project Root:", PROJECT_ROOT)
print("Python Version:", sys.version.split()[0])
print("Repository:", GITHUB_REPOSITORY)
print("API Version:", GITHUB_API_VERSION)

print("Authentication Configured:", bool(GITHUB_TOKEN))

print("\nHistorical Configuration:")

print("Reporting End:", AS_OF_TIMESTAMP)
print("Historical Cutoff:", HISTORICAL_CUTOFF)
print("Profiling Windows:", PROFILE_WINDOWS_DAYS)
print("Records Per Page:", PAGE_SIZE)
print("Maximum Pages:", MAX_PAGES)
print("Minimum API Reserve:", MIN_REMAINING_REQUESTS)

print("\nConfiguration Validation: Successful")

EIP | Historical Data Profiling
Project Root: /Users/khalid/Projects/Engineering-Intelligence-Platform
Python Version: 3.14.10
Repository: apache/airflow
API Version: 2026-03-10
Authentication Configured: True

Historical Configuration:
Reporting End: 2026-11-26 00:00:00+00:00
Historical Cutoff: 2026-08-28 00:00:00+00:00
Profiling Windows: [30, 60, 90]
Records Per Page: 100
Maximum Pages: 50
Minimum API Reserve: 500

Configuration Validation: Successful


In [4]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 02: Historical Pull Request Extraction
#
# Source ID: SRC-002
# Source System: GitHub REST API
# Source Entity: Pull Requests
# Source Endpoint: GET /repos/{owner}/{repo}/pulls
#
# Grain:
# One source observation per pull request.
#
# Purpose:
# Retrieve historical pull request candidates using bounded
# pagination, source update timestamps and configurable
# historical coverage requirements.
#
# Extraction Strategy:
# - Retrieve all pull request lifecycle states.
# - Request records by descending update timestamp.
# - Monitor source ordering and pagination consistency.
# - Confirm historical extraction boundaries.
# - Deduplicate records using stable GitHub identifiers.
# - Preserve original source payloads.
#
# Outputs:
# - historical_pull_requests_raw
# - historical_pull_requests_df
# - historical_candidates_df
# - pagination_summary_df
# - historical_coverage_summary_df
# - historical_scan_complete
#
# Limitations:
# GitHub pagination operates against a live source.
# Reaching a historical boundary does not guarantee a
# transactionally consistent or exhaustive snapshot.
# ============================================================


# ------------------------------------------------------------
# 01. Initialize extraction configuration
# ------------------------------------------------------------

# Require consecutive pages containing only records older
# than the historical cutoff before stopping extraction.
#
# This reduces the risk of premature termination when
# source records are not perfectly ordered.

MIN_BOUNDARY_CONFIRMATION_PAGES = 2

# Evaluate shorter diagnostic windows in addition to the
# primary historical profiling windows defined in Cell 02.

COVERAGE_WINDOWS_DAYS = sorted(
    set([7, 14, *PROFILE_WINDOWS_DAYS])
)

coverage_cutoffs = {
    window_days: (
        AS_OF_TIMESTAMP
        - pd.Timedelta(days=window_days)
    )
    for window_days in COVERAGE_WINDOWS_DAYS
}

# Track consecutive pages outside each historical window.

boundary_confirmation_counts = {
    window_days: 0
    for window_days in COVERAGE_WINDOWS_DAYS
}

# Record the page on which each boundary was confirmed.

boundary_confirmation_pages = {
    window_days: None
    for window_days in COVERAGE_WINDOWS_DAYS
}


# ------------------------------------------------------------
# 02. Initialize extraction containers
# ------------------------------------------------------------

historical_pull_requests_raw = []
pagination_results = []

historical_scan_complete = False

extraction_stop_reason = "MAXIMUM_PAGES_REACHED"

remaining_api_requests = None

ordering_warning_count = 0
cross_page_warning_count = 0

previous_page_oldest_updated_at = None

pull_requests_endpoint = (
    f"/repos/{GITHUB_REPOSITORY}/pulls"
)

extraction_started_at = pd.Timestamp.now(tz="UTC")

extraction_start_time = time.perf_counter()

print("EIP | Historical Pull Request Extraction")
print("=" * 75)

print("Repository:", GITHUB_REPOSITORY)
print("Historical Cutoff:", HISTORICAL_CUTOFF)
print("Maximum Pages:", MAX_PAGES)
print("Records Per Page:", PAGE_SIZE)
print("Coverage Windows:", COVERAGE_WINDOWS_DAYS)

print("\nStarting historical extraction...\n")


# ------------------------------------------------------------
# 03. Establish reusable HTTP connection
# ------------------------------------------------------------

# Reuse a single HTTP client throughout pagination to
# minimize repeated connection establishment overhead.

with httpx.Client(
    base_url=GITHUB_BASE_URL,
    headers=REQUEST_HEADERS,
    timeout=REQUEST_TIMEOUT_SECONDS,
    follow_redirects=False,
) as client:

    for page_number in range(1, MAX_PAGES + 1):

        # ----------------------------------------------------
        # 03.01. Configure paginated source request
        # ----------------------------------------------------

        request_params = {
            "state": "all",
            "sort": "updated",
            "direction": "desc",
            "per_page": PAGE_SIZE,
            "page": page_number,
        }


        # ----------------------------------------------------
        # 03.02. Retrieve source page
        # ----------------------------------------------------

        response = client.get(
            pull_requests_endpoint,
            params=request_params,
        )

        # Capture primary API quota information when available.

        rate_limit_remaining = response.headers.get(
            "x-ratelimit-remaining"
        )

        if rate_limit_remaining is not None:

            remaining_api_requests = int(
                rate_limit_remaining
            )


        # ----------------------------------------------------
        # 03.03. Handle rate-limit responses
        # ----------------------------------------------------

        # Preserve previously retrieved records if the source
        # explicitly indicates a rate-limit condition.
        #
        # Unexpected HTTP failures must still raise an error.

        is_rate_limited = (
            response.status_code == 429
            or (
                response.status_code == 403
                and (
                    remaining_api_requests == 0
                    or "retry-after" in response.headers
                    or "rate limit" in response.text.lower()
                )
            )
        )

        if is_rate_limited:

            extraction_stop_reason = "API_RATE_LIMIT_REACHED"

            print(
                f"WARNING: Rate limit encountered "
                f"on page {page_number:02d}."
            )

            print(
                "Retry-After:",
                response.headers.get("retry-after"),
            )

            print(
                "Rate Limit Reset:",
                response.headers.get("x-ratelimit-reset"),
            )

            break

        response.raise_for_status()


        # ----------------------------------------------------
        # 03.04. Validate source response structure
        # ----------------------------------------------------

        page_payload = response.json()

        if not isinstance(page_payload, list):

            raise ValueError(
                f"Unexpected source response structure "
                f"on page {page_number}."
            )

        if not all(
            isinstance(record, dict)
            for record in page_payload
        ):

            raise ValueError(
                f"Invalid source records on page {page_number}."
            )


        # ----------------------------------------------------
        # 03.05. Handle source exhaustion
        # ----------------------------------------------------

        if not page_payload:

            historical_scan_complete = True
            extraction_stop_reason = "SOURCE_EXHAUSTED"

            pagination_results.append(
                {
                    "pageNumber": page_number,
                    "recordsRetrieved": 0,
                    "oldestUpdatedAt": pd.NaT,
                    "latestUpdatedAt": pd.NaT,
                    "isPageOrdered": None,
                    "hasCrossPageOverlap": None,
                    "remainingApiRequests": (
                        remaining_api_requests
                    ),
                    "hasNextPage": False,
                }
            )

            print(
                f"Page {page_number:02d} | "
                "No additional records."
            )

            break


        # ----------------------------------------------------
        # 03.06. Validate source update timestamps
        # ----------------------------------------------------

        page_update_timestamps = pd.to_datetime(
            [
                record.get("updated_at")
                for record in page_payload
            ],
            utc=True,
            errors="raise",
        )

        if page_update_timestamps.isna().any():

            raise ValueError(
                f"Missing source update timestamps "
                f"on page {page_number}."
            )

        oldest_updated_at = page_update_timestamps.min()

        latest_updated_at = page_update_timestamps.max()


        # ----------------------------------------------------
        # 03.07. Assess ordering within the current page
        # ----------------------------------------------------

        # Unexpected ordering is recorded as a source quality
        # warning rather than terminating the extraction.

        is_page_ordered = bool(
            page_update_timestamps.is_monotonic_decreasing
        )

        if not is_page_ordered:

            ordering_warning_count += 1

            print(
                f"WARNING: Page {page_number:02d} "
                "contains out-of-order update timestamps."
            )


        # ----------------------------------------------------
        # 03.08. Assess ordering across consecutive pages
        # ----------------------------------------------------

        # In an ideally ordered descending result set,
        # the newest timestamp on the current page should
        # not exceed the oldest timestamp on the prior page.
        #
        # Equal timestamps are permitted.

        has_cross_page_overlap = (
            previous_page_oldest_updated_at is not None
            and latest_updated_at
            > previous_page_oldest_updated_at
        )

        if has_cross_page_overlap:

            cross_page_warning_count += 1

            print(
                f"WARNING: Page {page_number:02d} "
                "overlaps the previous page's timestamp range."
            )

        previous_page_oldest_updated_at = oldest_updated_at


        # ----------------------------------------------------
        # 03.09. Preserve original source payloads
        # ----------------------------------------------------

        # Do not discard records because of source ordering
        # inconsistencies. Deduplication and sorting will
        # occur after extraction.

        historical_pull_requests_raw.extend(
            page_payload
        )


        # ----------------------------------------------------
        # 03.10. Inspect pagination metadata
        # ----------------------------------------------------

        has_next_page = "next" in response.links

        pagination_results.append(
            {
                "pageNumber": page_number,
                "recordsRetrieved": len(page_payload),
                "oldestUpdatedAt": oldest_updated_at,
                "latestUpdatedAt": latest_updated_at,
                "isPageOrdered": is_page_ordered,
                "hasCrossPageOverlap": (
                    has_cross_page_overlap
                ),
                "remainingApiRequests": (
                    remaining_api_requests
                ),
                "hasNextPage": has_next_page,
            }
        )

        print(
            f"Page {page_number:02d} | "
            f"Records: {len(page_payload):3d} | "
            f"Oldest update: {oldest_updated_at} | "
            f"Remaining: {remaining_api_requests}"
        )


        # ----------------------------------------------------
        # 03.11. Evaluate historical coverage boundaries
        # ----------------------------------------------------

        # A page is entirely outside a historical window
        # only when its newest update precedes the cutoff.
        #
        # Using the newest timestamp avoids stopping when
        # a page contains both eligible and older records.

        for window_days, cutoff in coverage_cutoffs.items():

            if latest_updated_at < cutoff:

                boundary_confirmation_counts[
                    window_days
                ] += 1

            else:

                # Reset the confirmation if newer eligible
                # records appear on a subsequent page.

                boundary_confirmation_counts[
                    window_days
                ] = 0

                boundary_confirmation_pages[
                    window_days
                ] = None

            if (
                boundary_confirmation_counts[window_days]
                >= MIN_BOUNDARY_CONFIRMATION_PAGES
                and boundary_confirmation_pages[
                    window_days
                ] is None
            ):

                boundary_confirmation_pages[
                    window_days
                ] = page_number

                print(
                    f"BOUNDARY CONFIRMED: "
                    f"{window_days}-day window "
                    f"at page {page_number:02d}"
                )


        # ----------------------------------------------------
        # 03.12. Evaluate source exhaustion
        # ----------------------------------------------------

        if not has_next_page:

            historical_scan_complete = True

            extraction_stop_reason = "SOURCE_EXHAUSTED"

            break


        # ----------------------------------------------------
        # 03.13. Evaluate maximum historical cutoff
        # ----------------------------------------------------

        maximum_window_days = max(
            PROFILE_WINDOWS_DAYS
        )

        if (
            boundary_confirmation_counts[
                maximum_window_days
            ]
            >= MIN_BOUNDARY_CONFIRMATION_PAGES
        ):

            historical_scan_complete = True

            extraction_stop_reason = (
                "HISTORICAL_CUTOFF_REACHED"
            )

            break


        # ----------------------------------------------------
        # 03.14. Protect the available API request budget
        # ----------------------------------------------------

        if (
            remaining_api_requests is not None
            and remaining_api_requests
            <= MIN_REMAINING_REQUESTS
        ):

            extraction_stop_reason = (
                "MINIMUM_API_RESERVE_REACHED"
            )

            print(
                "Configured API reserve reached. "
                "Stopping historical extraction."
            )

            break


        # ----------------------------------------------------
        # 03.15. Apply request pacing
        # ----------------------------------------------------

        # Sequential request pacing reduces API pressure.
        # This does not guarantee avoidance of secondary
        # rate limits.

        if page_number < MAX_PAGES:

            time.sleep(REQUEST_DELAY_SECONDS)


# ------------------------------------------------------------
# 04. Capture extraction execution metadata
# ------------------------------------------------------------

extraction_completed_at = pd.Timestamp.now(
    tz="UTC"
)

extraction_duration_seconds = (
    time.perf_counter()
    - extraction_start_time
)

if not historical_pull_requests_raw:

    raise RuntimeError(
        "No pull requests retrieved for historical profiling."
    )


# ------------------------------------------------------------
# 05. Normalize original source records
# ------------------------------------------------------------

# Flatten nested JSON for exploration while preserving the
# original source payloads for future Bronze processing.

historical_pull_requests_df = pd.json_normalize(
    historical_pull_requests_raw,
    sep=".",
)

raw_record_count = len(
    historical_pull_requests_df
)

discovered_attribute_count = len(
    historical_pull_requests_df.columns
)


# ------------------------------------------------------------
# 06. Validate required source attributes
# ------------------------------------------------------------

required_columns = {
    "id",
    "number",
    "state",
    "draft",
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
}

missing_columns = (
    required_columns
    - set(historical_pull_requests_df.columns)
)

if missing_columns:

    raise ValueError(
        f"Required source attributes missing: "
        f"{sorted(missing_columns)}"
    )


# ------------------------------------------------------------
# 07. Preserve source identifier precision
# ------------------------------------------------------------

# GitHub pull request identifiers exceed the signed
# 32-bit integer range.
#
# Construct identifiers directly from the original JSON
# records to preserve their precision.

for identifier_column in ["id", "number"]:

    identifier_values = [
        record.get(identifier_column)
        for record in historical_pull_requests_raw
    ]

    if any(
        type(value) is not int or value <= 0
        for value in identifier_values
    ):

        raise ValueError(
            f"Invalid source identifier: "
            f"{identifier_column}"
        )

    historical_pull_requests_df[
        identifier_column
    ] = pd.array(
        identifier_values,
        dtype="Int64",
    )


# ------------------------------------------------------------
# 08. Standardize lifecycle timestamps
# ------------------------------------------------------------

timestamp_columns = [
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
]

for column in timestamp_columns:

    historical_pull_requests_df[column] = pd.to_datetime(
        historical_pull_requests_df[column],
        utc=True,
        errors="raise",
    )

# Creation and update timestamps are mandatory.
# Closure and merge timestamps may legitimately be null.

if historical_pull_requests_df["created_at"].isna().any():

    raise ValueError(
        "Missing pull request creation timestamps."
    )

if historical_pull_requests_df["updated_at"].isna().any():

    raise ValueError(
        "Missing pull request update timestamps."
    )



EIP | Historical Pull Request Extraction
Repository: apache/airflow
Historical Cutoff: 2026-08-28 00:00:00+00:00
Maximum Pages: 50
Records Per Page: 100
Coverage Windows: [7, 14, 30, 60, 90]

Starting historical extraction...

Page 01 | Records: 100 | Oldest update: 2026-11-25 19:47:50+00:00 | Remaining: 4899
Page 02 | Records: 100 | Oldest update: 2026-11-25 11:11:57+00:00 | Remaining: 4898
Page 03 | Records: 100 | Oldest update: 2026-11-24 19:59:39+00:00 | Remaining: 4897
Page 04 | Records: 100 | Oldest update: 2026-11-24 03:04:59+00:00 | Remaining: 4896
Page 05 | Records: 100 | Oldest update: 2026-11-22 16:15:28+00:00 | Remaining: 4895
Page 06 | Records: 100 | Oldest update: 2026-11-20 20:15:23+00:00 | Remaining: 4894
Page 07 | Records: 100 | Oldest update: 2026-11-19 20:08:10+00:00 | Remaining: 4893
Page 08 | Records: 100 | Oldest update: 2026-11-18 16:20:17+00:00 | Remaining: 4892
Page 09 | Records: 100 | Oldest update: 2026-11-17 09:35:48+00:00 | Remaining: 4891
Page 10 | Records

In [5]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 02B: Historical Source Validation and Coverage
#
# Source ID: SRC-002
# Source System: GitHub REST API
# Source Entity: Pull Requests
#
# Purpose:
# Validate historical source observations, quarantine
# invalid records, deduplicate pull requests and assess
# historical extraction coverage.
#
# Dependencies:
# Section 02A: Historical Pull Request Extraction
#
# Outputs:
# - historical_quarantine_df
# - historical_pull_requests_df
# - historical_candidates_df
# - pagination_summary_df
# - historical_coverage_summary_df
# - historical_data_quality_summary_df
#
# Processing Strategy:
# - Preserve original source payloads.
# - Isolate invalid source observations.
# - Continue processing valid observations.
# - Explicitly disclose historical coverage limitations.
#
# Note:
# This cell does not execute additional API requests.
# ============================================================


# ------------------------------------------------------------
# 01. Validate extraction dependencies
# ------------------------------------------------------------

if "historical_pull_requests_raw" not in globals():
    raise RuntimeError(
        "Historical source extraction has not been executed."
    )

if not historical_pull_requests_raw:
    raise RuntimeError(
        "No historical source observations are available."
    )

print("EIP | Historical Source Validation")
print("=" * 75)

print(
    "Raw Source Observations:",
    len(historical_pull_requests_raw),
)

print("Additional API Requests Required: 0")


# ------------------------------------------------------------
# 02. Reconstruct source dataset
# ------------------------------------------------------------

# Reconstruct from the original JSON payloads so that
# this validation cell can be rerun independently.
#
# The original source records remain unchanged.

historical_normalized_df = pd.json_normalize(
    historical_pull_requests_raw,
    sep=".",
)

raw_record_count = len(
    historical_normalized_df
)

discovered_attribute_count = len(
    historical_normalized_df.columns
)


# ------------------------------------------------------------
# 03. Validate required source attributes
# ------------------------------------------------------------

required_columns = {
    "id",
    "number",
    "state",
    "draft",
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
}

missing_columns = (
    required_columns
    - set(historical_normalized_df.columns)
)

if missing_columns:

    raise ValueError(
        f"Required source attributes missing: "
        f"{sorted(missing_columns)}"
    )


# ------------------------------------------------------------
# 04. Preserve source identifier precision
# ------------------------------------------------------------

# Construct nullable integer columns directly from
# original JSON records to avoid floating-point conversion.

for identifier_column in ["id", "number"]:

    identifier_values = [
        record.get(identifier_column)
        for record in historical_pull_requests_raw
    ]

    if any(
        type(value) is not int or value <= 0
        for value in identifier_values
    ):

        raise ValueError(
            f"Invalid source identifier: "
            f"{identifier_column}"
        )

    historical_normalized_df[
        identifier_column
    ] = pd.array(
        identifier_values,
        dtype="Int64",
    )


# ------------------------------------------------------------
# 05. Standardize lifecycle timestamps
# ------------------------------------------------------------

timestamp_columns = [
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
]

for column in timestamp_columns:

    historical_normalized_df[column] = pd.to_datetime(
        historical_normalized_df[column],
        utc=True,
        errors="raise",
    )


# ------------------------------------------------------------
# 06. Identify temporal data quality issues
# ------------------------------------------------------------

# Required lifecycle timestamps must be populated.

missing_required_timestamp_mask = (
    historical_normalized_df["created_at"].isna()
    | historical_normalized_df["updated_at"].isna()
)

# Update timestamps must not precede PR creation.

invalid_update_mask = (
    historical_normalized_df["updated_at"]
    < historical_normalized_df["created_at"]
)

# Closure timestamps must not precede PR creation.

invalid_closure_mask = (
    historical_normalized_df["closed_at"].notna()
    & (
        historical_normalized_df["closed_at"]
        < historical_normalized_df["created_at"]
    )
)

# Merge timestamps must not precede PR creation.

invalid_merge_mask = (
    historical_normalized_df["merged_at"].notna()
    & (
        historical_normalized_df["merged_at"]
        < historical_normalized_df["created_at"]
    )
)

# Consolidate all temporal validation failures.

invalid_timestamp_mask = (
    missing_required_timestamp_mask
    | invalid_update_mask
    | invalid_closure_mask
    | invalid_merge_mask
)


# ------------------------------------------------------------
# 07. Generate temporal data quality summary
# ------------------------------------------------------------

historical_data_quality_summary_df = pd.DataFrame(
    [
        {
            "ruleId": "DQ-H001",
            "ruleName": "Required timestamp completeness",
            "recordsChecked": raw_record_count,
            "failedRecords": int(
                missing_required_timestamp_mask.sum()
            ),
        },
        {
            "ruleId": "DQ-H002",
            "ruleName": "Update timestamp consistency",
            "recordsChecked": raw_record_count,
            "failedRecords": int(
                invalid_update_mask.sum()
            ),
        },
        {
            "ruleId": "DQ-H003",
            "ruleName": "Closure timestamp consistency",
            "recordsChecked": raw_record_count,
            "failedRecords": int(
                invalid_closure_mask.sum()
            ),
        },
        {
            "ruleId": "DQ-H004",
            "ruleName": "Merge timestamp consistency",
            "recordsChecked": raw_record_count,
            "failedRecords": int(
                invalid_merge_mask.sum()
            ),
        },
    ]
)

historical_data_quality_summary_df[
    "validationStatus"
] = (
    historical_data_quality_summary_df[
        "failedRecords"
    ]
    .map(
        lambda count: (
            "PASS"
            if count == 0
            else "FAIL"
        )
    )
)

print("\nEIP | Historical Data Quality Summary")
print("=" * 75)

display(historical_data_quality_summary_df)


# ------------------------------------------------------------
# 08. Quarantine invalid source observations
# ------------------------------------------------------------

# Preserve invalid records separately for investigation.
# Do not modify their original timestamp values.
#
# The original raw records remain available for future
# Bronze layer persistence.

historical_quarantine_df = (
    historical_normalized_df.loc[
        invalid_timestamp_mask
    ]
    .copy()
)

quarantined_record_count = len(
    historical_quarantine_df
)

# Attach individual validation failure indicators.

historical_quarantine_df[
    "hasMissingRequiredTimestamp"
] = missing_required_timestamp_mask.loc[
    historical_quarantine_df.index
]

historical_quarantine_df[
    "hasInvalidUpdateTimestamp"
] = invalid_update_mask.loc[
    historical_quarantine_df.index
]

historical_quarantine_df[
    "hasInvalidClosureTimestamp"
] = invalid_closure_mask.loc[
    historical_quarantine_df.index
]

historical_quarantine_df[
    "hasInvalidMergeTimestamp"
] = invalid_merge_mask.loc[
    historical_quarantine_df.index
]

# Calculate the difference between the source creation
# and update timestamps for investigation.

historical_quarantine_df[
    "updateMinusCreationSeconds"
] = (
    (
        historical_quarantine_df["updated_at"]
        - historical_quarantine_df["created_at"]
    )
    .dt.total_seconds()
)

print("\nEIP | Quarantined Source Observations")
print("=" * 75)

print(
    "Quarantined Records:",
    quarantined_record_count,
)

quarantine_display_columns = [
    "id",
    "number",
    "state",
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
    "updateMinusCreationSeconds",
    "hasMissingRequiredTimestamp",
    "hasInvalidUpdateTimestamp",
    "hasInvalidClosureTimestamp",
    "hasInvalidMergeTimestamp",
    "html_url",
]

if not historical_quarantine_df.empty:

    display(
        historical_quarantine_df.reindex(
            columns=quarantine_display_columns
        )
    )

else:

    print("No source observations quarantined.")


# ------------------------------------------------------------
# 09. Isolate valid historical source observations
# ------------------------------------------------------------

historical_valid_observations_df = (
    historical_normalized_df.loc[
        ~invalid_timestamp_mask
    ]
    .copy()
)

valid_observation_count = len(
    historical_valid_observations_df
)

if historical_valid_observations_df.empty:

    raise RuntimeError(
        "No valid historical observations remain "
        "after data quality validation."
    )


# ------------------------------------------------------------
# 10. Sort and deduplicate valid observations
# ------------------------------------------------------------

# Preserve the latest valid observation for each PR.
#
# If multiple observations have the same update timestamp,
# retain the later retrieved observation.

historical_pull_requests_df = (
    historical_valid_observations_df
    .sort_values(
        "updated_at",
        ascending=True,
        kind="stable",
    )
    .drop_duplicates(
        subset=["id"],
        keep="last",
    )
    .sort_values(
        ["updated_at", "id"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

duplicate_record_count = (
    valid_observation_count
    - len(historical_pull_requests_df)
)

if historical_pull_requests_df["number"].duplicated().any():

    raise ValueError(
        "Duplicate repository-specific pull request numbers."
    )


# ------------------------------------------------------------
# 11. Identify unresolved quarantined entities
# ------------------------------------------------------------

# A quarantined observation does not necessarily mean the
# entire PR is unavailable. Another valid observation of
# the same PR may exist in the extracted source records.

quarantined_pr_ids = set(
    historical_quarantine_df["id"]
)

valid_pr_ids = set(
    historical_pull_requests_df["id"]
)

unresolved_quarantined_pr_ids = (
    quarantined_pr_ids - valid_pr_ids
)

unresolved_quarantined_pr_count = len(
    unresolved_quarantined_pr_ids
)

print("\nEIP | Quarantine Impact Assessment")
print("=" * 75)

print(
    "Unique Quarantined Pull Requests:",
    len(quarantined_pr_ids),
)

print(
    "Quarantined PRs With Valid Alternative Observations:",
    len(quarantined_pr_ids & valid_pr_ids),
)

print(
    "Unresolved Quarantined Pull Requests:",
    unresolved_quarantined_pr_count,
)


# ------------------------------------------------------------
# 12. Reconcile source observation counts
# ------------------------------------------------------------

# Every extracted observation must be accounted for:
#
# Raw Observations =
#     Quarantined Observations
#   + Duplicate Valid Observations
#   + Unique Valid Pull Requests

reconciled_record_count = (
    quarantined_record_count
    + duplicate_record_count
    + len(historical_pull_requests_df)
)

record_reconciliation_passed = (
    raw_record_count == reconciled_record_count
)

if not record_reconciliation_passed:

    raise RuntimeError(
        "Historical source record reconciliation failed."
    )

historical_record_reconciliation_df = pd.DataFrame(
    [
        {
            "processingStage": "Raw source observations",
            "recordCount": raw_record_count,
        },
        {
            "processingStage": "Quarantined observations",
            "recordCount": quarantined_record_count,
        },
        {
            "processingStage": "Valid source observations",
            "recordCount": valid_observation_count,
        },
        {
            "processingStage": "Duplicate valid observations",
            "recordCount": duplicate_record_count,
        },
        {
            "processingStage": "Unique valid pull requests",
            "recordCount": len(
                historical_pull_requests_df
            ),
        },
    ]
)

print("\nEIP | Historical Record Reconciliation")
print("=" * 75)

display(historical_record_reconciliation_df)

print(
    "Record Count Reconciliation Passed:",
    record_reconciliation_passed,
)


# ------------------------------------------------------------
# 13. Identify historical ingestion candidates
# ------------------------------------------------------------

# Retain valid PRs whose latest observed update falls
# on or after the maximum historical cutoff.

historical_candidates_df = (
    historical_pull_requests_df.loc[
        historical_pull_requests_df["updated_at"]
        >= HISTORICAL_CUTOFF
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Consolidate pagination diagnostics
# ------------------------------------------------------------

pagination_summary_df = pd.DataFrame(
    pagination_results
)

print("\nEIP | Historical Pagination Summary")
print("=" * 75)

display(
    pagination_summary_df.tail(10)
)

print(
    "Total Pages Retrieved:",
    len(pagination_summary_df),
)

print(
    "Pages With Ordering Warnings:",
    ordering_warning_count,
)

print(
    "Pages With Cross-Page Overlap:",
    cross_page_warning_count,
)


# ------------------------------------------------------------
# 15. Assess historical coverage by reporting window
# ------------------------------------------------------------

# Boundary confirmation indicates estimated pagination
# coverage. It does not guarantee an exhaustive snapshot.
#
# Candidate counts include only valid, deduplicated PRs.

historical_coverage_records = []

source_exhausted = (
    extraction_stop_reason == "SOURCE_EXHAUSTED"
)

for window_days, cutoff in coverage_cutoffs.items():

    boundary_page = boundary_confirmation_pages[
        window_days
    ]

    window_boundary_reached = (
        source_exhausted
        or boundary_page is not None
    )

    candidate_count = int(
        historical_pull_requests_df["updated_at"]
        .ge(cutoff)
        .sum()
    )

    if source_exhausted:

        coverage_status = "SOURCE_EXHAUSTED"

    elif boundary_page is not None:

        coverage_status = "BOUNDARY_CONFIRMED_ESTIMATED"

    else:

        coverage_status = "INCOMPLETE_COVERAGE"

    historical_coverage_records.append(
        {
            "windowDays": window_days,
            "historicalCutoff": cutoff,
            "candidatePullRequests": candidate_count,
            "boundaryConfirmationPage": boundary_page,
            "windowBoundaryReached": (
                window_boundary_reached
            ),
            "coverageStatus": coverage_status,
        }
    )

historical_coverage_summary_df = pd.DataFrame(
    historical_coverage_records
)

historical_coverage_summary_df[
    "boundaryConfirmationPage"
] = historical_coverage_summary_df[
    "boundaryConfirmationPage"
].astype("Int64")

print("\nEIP | Historical Coverage Assessment")
print("=" * 75)

display(historical_coverage_summary_df)


# ------------------------------------------------------------
# 16. Generate historical source preview
# ------------------------------------------------------------

historical_preview_attributes = [
    "id",
    "number",
    "state",
    "draft",
    "user.id",
    "user.login",
    "user.type",
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
    "author_association",
    "base.ref",
    "head.ref",
    "html_url",
]

historical_pull_requests_preview_df = (
    historical_candidates_df.reindex(
        columns=historical_preview_attributes
    )
    .copy()
)

print("\nEIP | Historical Pull Request Preview")
print("=" * 75)

display(
    historical_pull_requests_preview_df.head(5)
)


# ------------------------------------------------------------
# 17. Generate source timestamp coverage summary
# ------------------------------------------------------------

oldest_created_at = (
    historical_pull_requests_df["created_at"].min()
)

latest_created_at = (
    historical_pull_requests_df["created_at"].max()
)

oldest_observed_update = (
    historical_pull_requests_df["updated_at"].min()
)

latest_observed_update = (
    historical_pull_requests_df["updated_at"].max()
)

observed_update_lookback_days = max(
    0.0,
    (
        AS_OF_TIMESTAMP
        - oldest_observed_update
    ).total_seconds() / 86400,
)

print("\nEIP | Source Timestamp Coverage")
print("=" * 75)

print("Oldest PR Creation:", oldest_created_at)
print("Latest PR Creation:", latest_created_at)

print(
    "Oldest Observed Update:",
    oldest_observed_update,
)

print(
    "Latest Observed Update:",
    latest_observed_update,
)

print(
    "Observed Update Lookback:",
    round(observed_update_lookback_days, 2),
    "days",
)


# ------------------------------------------------------------
# 18. Generate historical candidate state distribution
# ------------------------------------------------------------

historical_state_summary_df = (
    historical_candidates_df["state"]
    .value_counts(dropna=False)
    .rename_axis("pullRequestState")
    .reset_index(name="recordCount")
)

print("\nEIP | Historical Candidate State Distribution")
print("=" * 75)

display(historical_state_summary_df)


# ------------------------------------------------------------
# 19. Generate compact final extraction summary
# ------------------------------------------------------------

print("\nEIP | FINAL HISTORICAL EXTRACTION SUMMARY")
print("=" * 75)

print("Repository:", GITHUB_REPOSITORY)

print("Pages Retrieved:", len(pagination_summary_df))

print("Raw Records Retrieved:", raw_record_count)

print(
    "Discovered Source Attributes:",
    discovered_attribute_count,
)

print(
    "Quarantined Source Observations:",
    quarantined_record_count,
)

print(
    "Unresolved Quarantined Pull Requests:",
    unresolved_quarantined_pr_count,
)

print(
    "Valid Source Observations:",
    valid_observation_count,
)

print(
    "Duplicate Valid Observations:",
    duplicate_record_count,
)

print(
    "Unique Valid Pull Requests:",
    len(historical_pull_requests_df),
)

print(
    "Historical Candidates:",
    len(historical_candidates_df),
)

print(
    "Pages With Ordering Warnings:",
    ordering_warning_count,
)

print(
    "Pages With Cross-Page Overlap:",
    cross_page_warning_count,
)

print(
    "Remaining API Requests:",
    remaining_api_requests,
)

print(
    "Extraction Stop Reason:",
    extraction_stop_reason,
)

print(
    "Maximum Historical Scan Boundary Reached:",
    historical_scan_complete,
)

print(
    "Record Count Reconciliation Passed:",
    record_reconciliation_passed,
)

print(
    "Transactionally Consistent Snapshot:",
    "Not Guaranteed",
)

print(
    "Additional API Requests Executed:",
    0,
)

print("Historical PR Details Extracted: False")
print("Historical Reviews Extracted: False")
print("Database Persistence Completed: False")

print("\nHistorical Coverage:")

print(
    historical_coverage_summary_df.to_string(
        index=False
    )
)

if quarantined_record_count > 0:

    print(
        "\nWARNING: Invalid source observations were "
        "excluded from analytical processing."
    )

if unresolved_quarantined_pr_count > 0:

    print(
        "WARNING: Some PRs have no valid source observation. "
        "Historical metrics may therefore be incomplete."
    )

print(
    "\nHistorical Validation Status:",
    "COMPLETED_WITH_QUARANTINE"
    if quarantined_record_count > 0
    else "COMPLETED",
)

print("=" * 75)

EIP | Historical Source Validation
Raw Source Observations: 5000
Additional API Requests Required: 0

EIP | Historical Data Quality Summary


,ruleId,ruleName,recordsChecked,failedRecords,validationStatus
0,DQ-H001,Required timestamp completeness,5000,0,PASS
1,DQ-H002,Update timestamp consistency,5000,1,FAIL
2,DQ-H003,Closure timestamp consistency,5000,1,FAIL
3,DQ-H004,Merge timestamp consistency,5000,1,FAIL



EIP | Quarantined Source Observations
Quarantined Records: 1


,id,number,state,created_at,updated_at,closed_at,merged_at,updateMinusCreationSeconds,hasMissingRequiredTimestamp,hasInvalidUpdateTimestamp,hasInvalidClosureTimestamp,hasInvalidMergeTimestamp,html_url
2378,5105888939,76202,closed,2026-11-01 09:54:42+00:00,2026-11-01 09:01:49+00:00,2026-11-01 09:01:47+00:00,2026-11-01 09:01:47+00:00,-3173.0,False,True,True,True,https://github.com/apache/airflow/pull/76202



EIP | Quarantine Impact Assessment
Unique Quarantined Pull Requests: 1
Quarantined PRs With Valid Alternative Observations: 0
Unresolved Quarantined Pull Requests: 1

EIP | Historical Record Reconciliation


,processingStage,recordCount
0,Raw source observations,5000
1,Quarantined observations,1
2,Valid source observations,4999
3,Duplicate valid observations,0
4,Unique valid pull requests,4999


... (5 outputs left out)

,id,number,state,draft,user.id,user.login,user.type,created_at,updated_at,closed_at,merged_at,author_association,base.ref,head.ref,html_url
0,5508789145,77633,open,False,85239014,FrancescoCastaldi,User,2026-11-26 11:01:00+00:00,2026-11-26 11:01:07+00:00,NaT,NaT,NONE,main,fix-xcom-set-rfc9457-error,https://github.com/apache/airflow/pull/77633
1,5494312066,77598,open,False,49793253,ahnazary,User,2026-11-25 15:29:40+00:00,2026-11-26 10:57:43+00:00,NaT,NaT,CONTRIBUTOR,main,fix-rtif-cleanup-attempt-history,https://github.com/apache/airflow/pull/77598
2,5308612233,77080,open,False,605277,uranusjr,User,2026-11-14 07:17:33+00:00,2026-11-26 10:54:25+00:00,NaT,NaT,MEMBER,main,yaml-task-mapping-adr,https://github.com/apache/airflow/pull/77080
3,5106950521,76220,open,False,21106830,Zim95,User,2026-11-01 11:35:23+00:00,2026-11-26 10:51:45+00:00,NaT,NaT,NONE,main,fix/70762-temporal-mapper-chaining,https://github.com/apache/airflow/pull/76220
4,5272088932,76967,open,True,68415893,jason810496,User,2026-11-12 05:28:41+00:00,2026-11-26 10:44:53+00:00,NaT,NaT,MEMBER,main,feature/lang-sdk/taskflow-stub-mapped-sdk-binders,https://github.com/apache/airflow/pull/76967



EIP | Source Timestamp Coverage
Oldest PR Creation: 2017-09-20 00:08:32+00:00
Latest PR Creation: 2026-11-26 11:01:00+00:00
Oldest Observed Update: 2026-09-23 19:56:56+00:00
Latest Observed Update: 2026-11-26 11:01:07+00:00
Observed Update Lookback: 63.17 days

EIP | Historical Candidate State Distribution


,pullRequestState,recordCount
0,closed,4038
1,open,961



EIP | FINAL HISTORICAL EXTRACTION SUMMARY
Repository: apache/airflow
Pages Retrieved: 50
Raw Records Retrieved: 5000
Discovered Source Attributes: 368
Quarantined Source Observations: 1
Unresolved Quarantined Pull Requests: 1
Valid Source Observations: 4999
Duplicate Valid Observations: 0
Unique Valid Pull Requests: 4999
Historical Candidates: 4999
Pages With Ordering Warnings: 29
Pages With Cross-Page Overlap: 13
Remaining API Requests: 4850
Extraction Stop Reason: MAXIMUM_PAGES_REACHED
Maximum Historical Scan Boundary Reached: False
Record Count Reconciliation Passed: True
Transactionally Consistent Snapshot: Not Guaranteed
Additional API Requests Executed: 0
Historical PR Details Extracted: False
Historical Reviews Extracted: False
Database Persistence Completed: False

Historical Coverage:
 windowDays          historicalCutoff  candidatePullRequests  boundaryConfirmationPage  windowBoundaryReached               coverageStatus
          7 2026-11-19 00:00:00+00:00                  

In [6]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 03: Historical Activity and Volume Profiling
#
# Source ID: SRC-002
# Source System: GitHub REST API
# Source Entity: Pull Requests
#
# Purpose:
# Profile observed historical lifecycle activity and
# determine efficient source enrichment requirements.
#
# Dependencies:
# Section 01: Historical Profiling Configuration
# Section 02A: Historical Pull Request Extraction
# Section 02B: Historical Source Validation and Coverage
#
# Outputs:
# - historical_window_datasets
# - historical_volume_summary_df
# - historical_ingestion_summary_df
# - historical_coverage_assessment_df
# - historical_enrichment_summary_df
#
# Reporting Convention:
# UTC timestamps.
# Inclusive start boundary.
# Exclusive end boundary.
# Current incomplete UTC day excluded from event counts.
#
# Limitations:
# Historical metrics are based on observed source records.
# Live pagination and quarantined source records prevent
# guaranteed historical completeness.
#
# Note:
# This cell performs in-memory analysis only.
# No additional API requests are executed.
# ============================================================


# ------------------------------------------------------------
# 01. Validate historical extraction dependencies
# ------------------------------------------------------------

required_variables = [
    "historical_pull_requests_df",
    "historical_quarantine_df",
    "historical_coverage_summary_df",
    "pagination_summary_df",
    "COVERAGE_WINDOWS_DAYS",
    "AS_OF_TIMESTAMP",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        f"Missing historical dependencies: {missing_variables}"
    )

if historical_pull_requests_df.empty:
    raise RuntimeError(
        "No validated historical pull requests available."
    )


# ------------------------------------------------------------
# 02. Define reporting window utility
# ------------------------------------------------------------

def within_reporting_window(
    timestamps: pd.Series,
    start_timestamp: pd.Timestamp,
    end_timestamp: pd.Timestamp,
) -> pd.Series:
    """
    Identify events within a half-open UTC reporting window.

    Start timestamp is inclusive.
    End timestamp is exclusive.
    Missing timestamps are excluded.
    """

    return (
        timestamps.notna()
        & timestamps.ge(start_timestamp)
        & timestamps.lt(end_timestamp)
    )


# ------------------------------------------------------------
# 03. Initialize profiling datasets
# ------------------------------------------------------------

# Use all validated and deduplicated source observations.
# Do not modify the original historical extraction dataset.

historical_source_df = (
    historical_pull_requests_df.copy()
)

historical_window_datasets = {}

historical_volume_records = []
historical_ingestion_records = []
historical_enrichment_records = []
historical_coverage_records = []

coverage_lookup = (
    historical_coverage_summary_df
    .set_index("windowDays")
    .to_dict(orient="index")
)

SECONDS_PER_HOUR = 3600


# ------------------------------------------------------------
# 04. Profile individual historical windows
# ------------------------------------------------------------

for window_days in COVERAGE_WINDOWS_DAYS:

    # --------------------------------------------------------
    # 04.01. Establish reporting boundaries
    # --------------------------------------------------------

    window_start = (
        AS_OF_TIMESTAMP
        - pd.Timedelta(days=window_days)
    )

    window_end = AS_OF_TIMESTAMP

    coverage_info = coverage_lookup[window_days]

    boundary_reached = bool(
        coverage_info["windowBoundaryReached"]
    )


    # --------------------------------------------------------
    # 04.02. Identify historical ingestion candidates
    # --------------------------------------------------------

    # Candidate selection is based on the latest observed
    # update timestamp, including current-day updates.

    candidate_mask = (
        historical_source_df["updated_at"]
        .ge(window_start)
    )

    window_df = (
        historical_source_df.loc[candidate_mask]
        .copy()
    )

    historical_window_datasets[window_days] = window_df

    candidate_count = len(window_df)


    # --------------------------------------------------------
    # 04.03. Identify observed lifecycle events
    # --------------------------------------------------------

    # Calculate event cohorts from all valid observations
    # rather than relying exclusively on candidate selection.
    #
    # This provides an additional check against potential
    # inconsistencies in the source update timestamps.

    opened_mask = within_reporting_window(
        historical_source_df["created_at"],
        window_start,
        window_end,
    )

    merged_mask = within_reporting_window(
        historical_source_df["merged_at"],
        window_start,
        window_end,
    )

    closed_mask = within_reporting_window(
        historical_source_df["closed_at"],
        window_start,
        window_end,
    )

    updated_mask = within_reporting_window(
        historical_source_df["updated_at"],
        window_start,
        window_end,
    )

    closed_without_merge_mask = (
        closed_mask
        & historical_source_df["merged_at"].isna()
    )

    lifecycle_event_mask = (
        opened_mask
        | merged_mask
        | closed_mask
    )


    # --------------------------------------------------------
    # 04.04. Assess older pull request activity
    # --------------------------------------------------------

    # Older PRs may contribute to current-period metrics
    # even when their creation date precedes the window.

    older_pr_mask = (
        historical_source_df["created_at"]
        .lt(window_start)
    )

    older_candidate_mask = (
        candidate_mask
        & older_pr_mask
    )

    older_merged_mask = (
        older_pr_mask
        & merged_mask
    )

    older_closed_mask = (
        older_pr_mask
        & closed_mask
    )


    # --------------------------------------------------------
    # 04.05. Calculate observed merge duration statistics
    # --------------------------------------------------------

    # Merge duration is calculated using the cohort of
    # PRs merged within the reporting period.

    merged_prs_df = historical_source_df.loc[
        merged_mask
    ].copy()

    merge_hours = (
        (
            merged_prs_df["merged_at"]
            - merged_prs_df["created_at"]
        )
        .dt.total_seconds()
        / SECONDS_PER_HOUR
    )

    valid_merge_hours = merge_hours.loc[
        merge_hours.notna()
        & merge_hours.ge(0)
    ]

    invalid_merge_duration_count = (
        len(merged_prs_df)
        - len(valid_merge_hours)
    )


    # --------------------------------------------------------
    # 04.06. Assess current source lifecycle states
    # --------------------------------------------------------

    # These represent current observed PR states.
    # They are not historical backlog snapshots.

    currently_open_mask = (
        window_df["state"].eq("open")
    )

    currently_closed_mask = (
        window_df["state"].eq("closed")
    )

    currently_draft_mask = (
        currently_open_mask
        & window_df["draft"].eq(True)
    )

    currently_open_non_draft_mask = (
        currently_open_mask
        & window_df["draft"].eq(False)
    )


    # --------------------------------------------------------
    # 04.07. Assess quarantined record impact
    # --------------------------------------------------------

    # Identify quarantined records with source lifecycle
    # timestamps falling inside this reporting window.
    #
    # These timestamps are unreliable and are used here
    # only to indicate potential data quality impact.

    quarantine_opened_mask = within_reporting_window(
        historical_quarantine_df["created_at"],
        window_start,
        window_end,
    )

    quarantine_merged_mask = within_reporting_window(
        historical_quarantine_df["merged_at"],
        window_start,
        window_end,
    )

    quarantine_closed_mask = within_reporting_window(
        historical_quarantine_df["closed_at"],
        window_start,
        window_end,
    )

    quarantine_event_mask = (
        quarantine_opened_mask
        | quarantine_merged_mask
        | quarantine_closed_mask
    )

    affected_quarantined_prs = (
        historical_quarantine_df.loc[
            quarantine_event_mask,
            "id",
        ]
        .nunique()
    )


    # --------------------------------------------------------
    # 04.08. Classify analytical coverage
    # --------------------------------------------------------

    # A confirmed boundary does not prove exhaustive
    # coverage when the upstream ordering is inconsistent.

    if not boundary_reached:

        analytical_coverage_status = (
            "PARTIAL_COVERAGE"
        )

    elif affected_quarantined_prs > 0:

        analytical_coverage_status = (
            "ESTIMATED_WITH_QUARANTINE"
        )

    else:

        analytical_coverage_status = (
            "ESTIMATED_COVERAGE"
        )


    # --------------------------------------------------------
    # 04.09. Store historical lifecycle measures
    # --------------------------------------------------------

    historical_volume_records.append(
        {
            "windowDays": window_days,
            "windowStart": window_start,
            "windowEndExclusive": window_end,
            "windowBoundaryReached": boundary_reached,
            "coverageStatus": analytical_coverage_status,

            "candidatePullRequests": candidate_count,

            "pullRequestsOpened": int(opened_mask.sum()),
            "pullRequestsMerged": int(merged_mask.sum()),
            "pullRequestsClosed": int(closed_mask.sum()),

            "closedWithoutMerge": int(
                closed_without_merge_mask.sum()
            ),

            "latestUpdateInWindow": int(
                updated_mask.sum()
            ),

            "medianMergeHours": valid_merge_hours.median(),
            "meanMergeHours": valid_merge_hours.mean(),
            "p90MergeHours": valid_merge_hours.quantile(0.90),

            "mergeDurationSampleSize": len(valid_merge_hours),

            "invalidMergeDurations": (
                invalid_merge_duration_count
            ),
        }
    )


    # --------------------------------------------------------
    # 04.10. Store ingestion characteristics
    # --------------------------------------------------------

    historical_ingestion_records.append(
        {
            "windowDays": window_days,
            "coverageStatus": analytical_coverage_status,
            "candidatePullRequests": candidate_count,

            "olderPullRequests": int(
                older_candidate_mask.sum()
            ),

            "olderPullRequestsMerged": int(
                older_merged_mask.sum()
            ),

            "olderPullRequestsClosed": int(
                older_closed_mask.sum()
            ),

            "currentlyOpen": int(
                currently_open_mask.sum()
            ),

            "currentlyClosed": int(
                currently_closed_mask.sum()
            ),

            "currentlyDraft": int(
                currently_draft_mask.sum()
            ),

            "currentlyOpenNonDraft": int(
                currently_open_non_draft_mask.sum()
            ),

            "currentDayCreated": int(
                window_df["created_at"].ge(window_end).sum()
            ),

            "currentDayUpdated": int(
                window_df["updated_at"].ge(window_end).sum()
            ),
        }
    )


    # --------------------------------------------------------
    # 04.11. Estimate alternative enrichment strategies
    # --------------------------------------------------------

    # Strategy A:
    # Enrich every candidate PR with details and reviews.
    #
    # Strategy B:
    # Enrich only PRs with observed creation, closure
    # or merge activity within the reporting period.
    #
    # Strategy C:
    # Enrich only PRs merged within the reporting period.
    # Suitable for completed lifecycle analysis.
    #
    # Strategy D:
    # Retrieve reviews only for merged PRs.
    # Suitable for review timing analysis without
    # code change size or detail-level statistics.
    #
    # Review requests assume a minimum of one page per PR.

    lifecycle_pr_count = int(
        lifecycle_event_mask.sum()
    )

    merged_pr_count = int(
        merged_mask.sum()
    )

    historical_enrichment_records.append(
        {
            "windowDays": window_days,

            "candidatePullRequests": candidate_count,

            "lifecycleEventPullRequests": (
                lifecycle_pr_count
            ),

            "mergedCohortPullRequests": (
                merged_pr_count
            ),

            "fullEnrichmentRequests": (
                candidate_count * 2
            ),

            "lifecycleEnrichmentRequests": (
                lifecycle_pr_count * 2
            ),

            "mergedCohortEnrichmentRequests": (
                merged_pr_count * 2
            ),

            "mergedCohortReviewOnlyRequests": (
                merged_pr_count
            ),
        }
    )


    # --------------------------------------------------------
    # 04.12. Store coverage and quality assessment
    # --------------------------------------------------------

    # Identify observed lifecycle events belonging to
    # PRs outside the updated-at candidate cohort.

    events_outside_candidates = int(
        (
            lifecycle_event_mask
            & ~candidate_mask
        ).sum()
    )

    historical_coverage_records.append(
        {
            "windowDays": window_days,

            "sourceCoverageStatus": (
                coverage_info["coverageStatus"]
            ),

            "analyticalCoverageStatus": (
                analytical_coverage_status
            ),

            "boundaryConfirmationPage": (
                coverage_info["boundaryConfirmationPage"]
            ),

            "windowBoundaryReached": boundary_reached,

            "quarantinedPRsInWindow": (
                affected_quarantined_prs
            ),

            "eventsOutsideCandidateCohort": (
                events_outside_candidates
            ),

            "extractionCandidateCount": (
                coverage_info["candidatePullRequests"]
            ),

            "profiledCandidateCount": candidate_count,

            "isCandidateCountReconciled": (
                candidate_count
                == coverage_info["candidatePullRequests"]
            ),

            "isCoverageGuaranteed": False,
        }
    )


# ------------------------------------------------------------
# 05. Consolidate profiling results
# ------------------------------------------------------------

historical_volume_summary_df = pd.DataFrame(
    historical_volume_records
)

historical_ingestion_summary_df = pd.DataFrame(
    historical_ingestion_records
)

historical_enrichment_summary_df = pd.DataFrame(
    historical_enrichment_records
)

historical_coverage_assessment_df = pd.DataFrame(
    historical_coverage_records
)


# ------------------------------------------------------------
# 06. Calculate derived activity measures
# ------------------------------------------------------------

# Observed PR creations minus observed closures.
# This is not an exact historical backlog change because
# reopening events are not available in the current dataset.

historical_volume_summary_df["netObservedPRInflow"] = (
    historical_volume_summary_df["pullRequestsOpened"]
    - historical_volume_summary_df["pullRequestsClosed"]
)

# Suppress daily averages for incomplete historical windows.

for source_column, target_column in [
    ("pullRequestsOpened", "avgDailyOpened"),
    ("pullRequestsMerged", "avgDailyMerged"),
]:

    historical_volume_summary_df[target_column] = (
        historical_volume_summary_df[source_column]
        .div(historical_volume_summary_df["windowDays"])
        .where(
            historical_volume_summary_df[
                "windowBoundaryReached"
            ]
        )
    )


# ------------------------------------------------------------
# 07. Calculate ingestion characteristics
# ------------------------------------------------------------

valid_candidate_counts = (
    historical_ingestion_summary_df["candidatePullRequests"]
    .where(
        historical_ingestion_summary_df[
            "candidatePullRequests"
        ].gt(0)
    )
)

historical_ingestion_summary_df["olderPullRequestPct"] = (
    historical_ingestion_summary_df["olderPullRequests"]
    .div(valid_candidate_counts)
    .mul(100)
)

historical_ingestion_summary_df["currentlyOpenPct"] = (
    historical_ingestion_summary_df["currentlyOpen"]
    .div(valid_candidate_counts)
    .mul(100)
)

# Maintain compatibility with subsequent ingestion
# feasibility calculations.

historical_ingestion_summary_df["estimatedDetailRequests"] = (
    historical_ingestion_summary_df["candidatePullRequests"]
)

historical_ingestion_summary_df["minimumReviewRequests"] = (
    historical_ingestion_summary_df["candidatePullRequests"]
)

historical_ingestion_summary_df["minimumEnrichmentRequests"] = (
    historical_ingestion_summary_df["estimatedDetailRequests"]
    + historical_ingestion_summary_df["minimumReviewRequests"]
)


# ------------------------------------------------------------
# 08. Calculate potential API request savings
# ------------------------------------------------------------

# Savings are relative to enriching every observed candidate.
# These estimates exclude list requests, repository metadata,
# additional review pagination and retries.

full_enrichment_baseline = (
    historical_enrichment_summary_df["fullEnrichmentRequests"]
    .where(
        historical_enrichment_summary_df[
            "fullEnrichmentRequests"
        ].gt(0)
    )
)

historical_enrichment_summary_df["mergedCohortSavingsPct"] = (
    1
    - historical_enrichment_summary_df[
        "mergedCohortEnrichmentRequests"
    ].div(full_enrichment_baseline)
).mul(100)

historical_enrichment_summary_df["reviewOnlySavingsPct"] = (
    1
    - historical_enrichment_summary_df[
        "mergedCohortReviewOnlyRequests"
    ].div(full_enrichment_baseline)
).mul(100)


# ------------------------------------------------------------
# 09. Validate historical profiling results
# ------------------------------------------------------------

historical_validation_results = {
    "Candidate counts reconciled": bool(
        historical_coverage_assessment_df[
            "isCandidateCountReconciled"
        ].all()
    ),

    "Candidate counts increase with lookback": bool(
        historical_ingestion_summary_df
        .sort_values("windowDays")["candidatePullRequests"]
        .is_monotonic_increasing
    ),

    "Current PR states reconciled": bool(
        (
            historical_ingestion_summary_df["currentlyOpen"]
            + historical_ingestion_summary_df["currentlyClosed"]
        ).eq(
            historical_ingestion_summary_df[
                "candidatePullRequests"
            ]
        ).all()
    ),

    "Open PR draft states reconciled": bool(
        (
            historical_ingestion_summary_df["currentlyDraft"]
            + historical_ingestion_summary_df[
                "currentlyOpenNonDraft"
            ]
        ).eq(
            historical_ingestion_summary_df["currentlyOpen"]
        ).all()
    ),

    "Merge durations valid": bool(
        historical_volume_summary_df[
            "invalidMergeDurations"
        ].eq(0).all()
    ),

    "Lifecycle events covered by candidate selection": bool(
        historical_coverage_assessment_df[
            "eventsOutsideCandidateCohort"
        ].eq(0).all()
    ),
}

historical_volume_validation_df = pd.DataFrame(
    [
        {
            "validationRule": rule,
            "validationStatus": "PASS" if passed else "FAIL",
        }
        for rule, passed in historical_validation_results.items()
    ]
)

all_volume_checks_passed = all(
    historical_validation_results.values()
)


# ------------------------------------------------------------
# 10. Display historical lifecycle activity
# ------------------------------------------------------------

print("EIP | Historical Pull Request Activity")
print("=" * 75)

display(
    historical_volume_summary_df[
        [
            "windowDays",
            "coverageStatus",
            "candidatePullRequests",
            "pullRequestsOpened",
            "pullRequestsMerged",
            "pullRequestsClosed",
            "closedWithoutMerge",
            "netObservedPRInflow",
        ]
    ]
)


# ------------------------------------------------------------
# 11. Display observed merge duration statistics
# ------------------------------------------------------------

print("\nEIP | Historical Merge Duration Statistics")
print("=" * 75)

display(
    historical_volume_summary_df[
        [
            "windowDays",
            "coverageStatus",
            "medianMergeHours",
            "meanMergeHours",
            "p90MergeHours",
            "mergeDurationSampleSize",
            "avgDailyOpened",
            "avgDailyMerged",
        ]
    ].round(2)
)


# ------------------------------------------------------------
# 12. Display historical ingestion characteristics
# ------------------------------------------------------------

print("\nEIP | Historical Ingestion Characteristics")
print("=" * 75)

display(
    historical_ingestion_summary_df[
        [
            "windowDays",
            "candidatePullRequests",
            "olderPullRequests",
            "olderPullRequestPct",
            "olderPullRequestsMerged",
            "currentlyOpen",
            "currentlyClosed",
            "currentlyDraft",
        ]
    ].round(2)
)


# ------------------------------------------------------------
# 13. Display alternative enrichment strategies
# ------------------------------------------------------------

print("\nEIP | Historical Enrichment Strategy Comparison")
print("=" * 75)

display(
    historical_enrichment_summary_df.round(2)
)


# ------------------------------------------------------------
# 14. Display coverage and quality assessment
# ------------------------------------------------------------

print("\nEIP | Historical Coverage and Quality")
print("=" * 75)

display(
    historical_coverage_assessment_df[
        [
            "windowDays",
            "analyticalCoverageStatus",
            "boundaryConfirmationPage",
            "quarantinedPRsInWindow",
            "eventsOutsideCandidateCohort",
            "isCandidateCountReconciled",
            "isCoverageGuaranteed",
        ]
    ]
)


# ------------------------------------------------------------
# 15. Display validation results
# ------------------------------------------------------------

print("\nEIP | Historical Profiling Validation")
print("=" * 75)

display(historical_volume_validation_df)


# ------------------------------------------------------------
# 16. Generate compact final profiling summary
# ------------------------------------------------------------

historical_final_summary_df = (
    historical_volume_summary_df[
        [
            "windowDays",
            "coverageStatus",
            "candidatePullRequests",
            "pullRequestsOpened",
            "pullRequestsMerged",
            "medianMergeHours",
        ]
    ]
    .merge(
        historical_enrichment_summary_df[
            [
                "windowDays",
                "fullEnrichmentRequests",
                "mergedCohortEnrichmentRequests",
                "mergedCohortReviewOnlyRequests",
            ]
        ],
        on="windowDays",
        how="left",
        validate="one_to_one",
    )
)

print("\nEIP | FINAL HISTORICAL PROFILING SUMMARY")
print("=" * 75)

print(
    historical_final_summary_df
    .round({"medianMergeHours": 2})
    .to_string(index=False)
)

print("\nValidated Unique Pull Requests:",
      len(historical_source_df))

print("Historical Volume Checks Passed:",
      all_volume_checks_passed)

print("Pages With Ordering Warnings:",
      ordering_warning_count)

print("Pages With Cross-Page Overlap:",
      cross_page_warning_count)

print("Quarantined Source Observations:",
      quarantined_record_count)

print("Additional API Requests Executed: 0")

print("Guaranteed Historical Coverage: False")

print("Historical PR Details Extracted: False")
print("Historical Reviews Extracted: False")
print("Database Persistence Completed: False")

print(
    "Historical Profiling Status:",
    "COMPLETED_WITH_LIMITATIONS"
    if all_volume_checks_passed
    else "COMPLETED_WITH_VALIDATION_FAILURES",
)

EIP | Historical Pull Request Activity


,windowDays,coverageStatus,candidatePullRequests,pullRequestsOpened,pullRequestsMerged,pullRequestsClosed,closedWithoutMerge,netObservedPRInflow
0,7,ESTIMATED_COVERAGE,773,305,208,300,92,5
1,14,ESTIMATED_COVERAGE,1219,578,430,570,140,8
2,30,ESTIMATED_WITH_QUARANTINE,2991,1717,1305,1676,371,41
3,60,ESTIMATED_WITH_QUARANTINE,4707,3449,2603,3325,722,124
4,90,PARTIAL_COVERAGE,4999,4045,2714,3659,945,386



EIP | Historical Merge Duration Statistics


,windowDays,coverageStatus,medianMergeHours,meanMergeHours,p90MergeHours,mergeDurationSampleSize,avgDailyOpened,avgDailyMerged
0,7,ESTIMATED_COVERAGE,66.53,252.09,532.73,208,43.57,29.71
1,14,ESTIMATED_COVERAGE,61.72,260.21,531.12,430,41.29,30.71
2,30,ESTIMATED_WITH_QUARANTINE,19.41,209.21,530.84,1305,57.23,43.50
3,60,ESTIMATED_WITH_QUARANTINE,20.18,214.93,556.68,2603,57.48,43.38
4,90,PARTIAL_COVERAGE,20.25,213.91,556.61,2714,NaN,NaN



EIP | Historical Ingestion Characteristics


,windowDays,candidatePullRequests,olderPullRequests,olderPullRequestPct,olderPullRequestsMerged,currentlyOpen,currentlyClosed,currentlyDraft
0,7,773,454,58.73,89,452,321,59
1,14,1219,627,51.44,167,615,604,80
2,30,2991,1260,42.13,220,833,2158,118
3,60,4707,1244,26.43,296,961,3746,137
4,90,4999,940,18.80,133,961,4038,137


... (3 outputs left out)

,windowDays,analyticalCoverageStatus,boundaryConfirmationPage,quarantinedPRsInWindow,eventsOutsideCandidateCohort,isCandidateCountReconciled,isCoverageGuaranteed
0,7,ESTIMATED_COVERAGE,10.0,0,0,True,False
1,14,ESTIMATED_COVERAGE,29.0,0,0,True,False
2,30,ESTIMATED_WITH_QUARANTINE,50.0,1,0,True,False
3,60,ESTIMATED_WITH_QUARANTINE,50.0,1,0,True,False
4,90,PARTIAL_COVERAGE,NaN,1,0,True,False



EIP | Historical Profiling Validation


,validationRule,validationStatus
0,Candidate counts reconciled,PASS
1,Candidate counts increase with lookback,PASS
2,Current PR states reconciled,PASS
3,Open PR draft states reconciled,PASS
4,Merge durations valid,PASS
5,Lifecycle events covered by candidate selection,PASS



EIP | FINAL HISTORICAL PROFILING SUMMARY
 windowDays            coverageStatus  candidatePullRequests  pullRequestsOpened  pullRequestsMerged  medianMergeHours  fullEnrichmentRequests  mergedCohortEnrichmentRequests  mergedCohortReviewOnlyRequests
          7        ESTIMATED_COVERAGE                    773                 305                 208             66.53                    1546                             416                             208
         14        ESTIMATED_COVERAGE                   1219                 578                 430             61.72                    2438                             860                             430
         30 ESTIMATED_WITH_QUARANTINE                   2991                1717                1305             19.41                    5982                            2610                            1305
         60 ESTIMATED_WITH_QUARANTINE                   4707                3449                2603             20.18            

In [7]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 04: Weekly Activity and Historical Benchmarks
#
# Source ID: SRC-002
# Source System: GitHub REST API
#
# Purpose:
# Generate non-overlapping historical reporting periods,
# calculate engineering throughput and merge duration,
# and establish preliminary historical benchmarks.
#
# Dependencies:
# Section 02A: Historical Pull Request Extraction
# Section 02B: Historical Source Validation
# Section 03: Historical Activity and Volume Profiling
#
# Outputs:
# - weekly_activity_df
# - rolling_weekly_activity_df
# - weekly_benchmark_df
# - weekly_validation_df
#
# Proposed Gold Target:
# eip_tbl_gold_periodFacts_long
#
# Reporting Convention:
# Calendar weeks: Monday to Sunday, UTC.
# Rolling periods: Seven complete UTC calendar days.
# Start boundary inclusive, end boundary exclusive.
#
# Limitations:
# Historical metrics represent observed source activity.
# Coverage is estimated and not guaranteed exhaustive.
#
# Note:
# This cell executes no additional GitHub API requests.
# ============================================================


# ------------------------------------------------------------
# 01. Configure reporting and benchmark parameters
# ------------------------------------------------------------

REPORTING_PERIOD_DAYS = 7

CALENDAR_WEEK_COUNT = 8

BENCHMARK_PERIOD_COUNT = 4

# One current period plus four historical benchmark periods.

ROLLING_PERIOD_COUNT = BENCHMARK_PERIOD_COUNT + 1

SECONDS_PER_HOUR = 3600

# Reuse the validated and deduplicated source dataset.

weekly_source_df = historical_pull_requests_df.copy()


# ------------------------------------------------------------
# 02. Define reporting period coverage assessment
# ------------------------------------------------------------

def assess_period_coverage(
    period_start: pd.Timestamp,
    period_end: pd.Timestamp,
) -> dict:
    """
    Assess estimated source coverage and known quarantine
    impact for an individual reporting period.

    Boundary confirmation is an extraction heuristic.
    It does not establish an exhaustive source snapshot.
    """

    eligible_coverage_df = (
        historical_coverage_summary_df.loc[
            historical_coverage_summary_df[
                "windowBoundaryReached"
            ]
            & historical_coverage_summary_df[
                "historicalCutoff"
            ].le(period_start)
        ]
        .sort_values("windowDays")
    )

    boundary_reached = not eligible_coverage_df.empty

    # Identify quarantined PRs whose unreliable lifecycle
    # timestamps suggest a potential impact on this period.

    quarantine_mask = pd.Series(
        False,
        index=historical_quarantine_df.index,
        dtype=bool,
    )

    for column in [
        "created_at",
        "closed_at",
        "merged_at",
    ]:

        quarantine_mask |= within_reporting_window(
            historical_quarantine_df[column],
            period_start,
            period_end,
        )

    affected_pr_count = int(
        historical_quarantine_df.loc[
            quarantine_mask,
            "id",
        ].nunique()
    )

    if not boundary_reached:

        coverage_status = "PARTIAL_COVERAGE"

    elif affected_pr_count > 0:

        coverage_status = "ESTIMATED_WITH_QUARANTINE"

    else:

        coverage_status = "ESTIMATED_COVERAGE"

    return {
        "windowBoundaryReached": boundary_reached,
        "coverageStatus": coverage_status,
        "quarantinedPRsInPeriod": affected_pr_count,
        "isCoverageGuaranteed": False,
    }


# ------------------------------------------------------------
# 03. Define reusable period metric calculation
# ------------------------------------------------------------

def calculate_period_activity(
    period_start: pd.Timestamp,
    period_end: pd.Timestamp,
    period_type: str,
) -> dict:
    """
    Calculate observed engineering activity for a single
    non-overlapping reporting period.

    Merge duration statistics use the cohort of PRs
    successfully merged within the reporting period.
    """

    # Identify lifecycle events independently.

    opened_mask = within_reporting_window(
        weekly_source_df["created_at"],
        period_start,
        period_end,
    )

    merged_mask = within_reporting_window(
        weekly_source_df["merged_at"],
        period_start,
        period_end,
    )

    closed_mask = within_reporting_window(
        weekly_source_df["closed_at"],
        period_start,
        period_end,
    )

    closed_without_merge_mask = (
        closed_mask
        & weekly_source_df["merged_at"].isna()
    )

    # Identify merged PRs created before the reporting period.

    older_merged_mask = (
        merged_mask
        & weekly_source_df["created_at"].lt(period_start)
    )

    # Calculate merge duration directly from source timestamps.

    merged_prs_df = weekly_source_df.loc[
        merged_mask
    ]

    merge_hours = (
        (
            merged_prs_df["merged_at"]
            - merged_prs_df["created_at"]
        )
        .dt.total_seconds()
        / SECONDS_PER_HOUR
    )

    valid_merge_hours = merge_hours.loc[
        merge_hours.notna()
        & merge_hours.ge(0)
    ]

    # Count observed PR creation, merge and closure events.

    prs_opened = int(opened_mask.sum())
    prs_merged = int(merged_mask.sum())
    prs_closed = int(closed_mask.sum())

    # Attach period-level source coverage metadata.

    coverage = assess_period_coverage(
        period_start,
        period_end,
    )

    return {
        "periodType": period_type,
        "periodStart": period_start,
        "periodEndExclusive": period_end,

        "pullRequestsOpened": prs_opened,
        "pullRequestsMerged": prs_merged,
        "pullRequestsClosed": prs_closed,

        "closedWithoutMerge": int(
            closed_without_merge_mask.sum()
        ),

        "netObservedPRInflow": (
            prs_opened - prs_closed
        ),

        "olderPullRequestsMerged": int(
            older_merged_mask.sum()
        ),

        "medianMergeHours": (
            valid_merge_hours.median()
        ),

        "meanMergeHours": (
            valid_merge_hours.mean()
        ),

        "p90MergeHours": (
            valid_merge_hours.quantile(0.90)
        ),

        "mergeDurationSampleSize": (
            len(valid_merge_hours)
        ),

        "invalidMergeDurations": (
            prs_merged - len(valid_merge_hours)
        ),

        **coverage,
    }


# ------------------------------------------------------------
# 04. Define period-over-period comparison logic
# ------------------------------------------------------------

def add_period_comparisons(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Calculate changes against the immediately preceding
    non-overlapping period.

    Percentage changes are suppressed when either period
    lacks estimated boundary coverage or the previous
    metric value is zero or missing.
    """

    result_df = (
        dataframe
        .sort_values("periodStart")
        .reset_index(drop=True)
        .copy()
    )

    comparison_metrics = {
        "pullRequestsOpened": "opened",
        "pullRequestsMerged": "merged",
        "pullRequestsClosed": "closed",
        "medianMergeHours": "medianMergeHours",
    }

    # Both reporting periods must have reached their
    # estimated historical coverage boundaries.

    comparable_period_mask = (
        result_df["windowBoundaryReached"]
        & result_df["windowBoundaryReached"].shift(
            1,
            fill_value=False,
        )
    )

    for source_column, metric_prefix in (
        comparison_metrics.items()
    ):

        previous_values = (
            result_df[source_column].shift(1)
        )

        current_values = (
            result_df[source_column]
        )

        result_df[f"{metric_prefix}WoWChange"] = (
            current_values
            .sub(previous_values)
            .where(comparable_period_mask)
        )

        result_df[f"{metric_prefix}WoWPct"] = (
            current_values
            .sub(previous_values)
            .div(
                previous_values.where(
                    previous_values.gt(0)
                )
            )
            .mul(100)
            .where(comparable_period_mask)
        )

    return result_df


# ------------------------------------------------------------
# 05. Generate complete calendar reporting weeks
# ------------------------------------------------------------

# Identify the start of the current incomplete calendar week.

CURRENT_WEEK_START = (
    AS_OF_TIMESTAMP
    - pd.Timedelta(
        days=AS_OF_TIMESTAMP.weekday()
    )
)

calendar_week_starts = pd.date_range(
    end=(
        CURRENT_WEEK_START
        - pd.Timedelta(days=REPORTING_PERIOD_DAYS)
    ),
    periods=CALENDAR_WEEK_COUNT,
    freq="7D",
)

calendar_week_records = []

for week_start in calendar_week_starts:

    week_end = (
        week_start
        + pd.Timedelta(days=REPORTING_PERIOD_DAYS)
    )

    calendar_week_records.append(
        calculate_period_activity(
            period_start=week_start,
            period_end=week_end,
            period_type="CALENDAR_WEEK",
        )
    )

weekly_activity_df = pd.DataFrame(
    calendar_week_records
)

weekly_activity_df = add_period_comparisons(
    weekly_activity_df
)


# ------------------------------------------------------------
# 06. Generate rolling seven-day reporting periods
# ------------------------------------------------------------

# Generate the latest complete seven-day period and four
# preceding, non-overlapping seven-day benchmark periods.

rolling_period_records = []

for period_offset in reversed(
    range(ROLLING_PERIOD_COUNT)
):

    period_end = (
        AS_OF_TIMESTAMP
        - pd.Timedelta(
            days=period_offset * REPORTING_PERIOD_DAYS
        )
    )

    period_start = (
        period_end
        - pd.Timedelta(days=REPORTING_PERIOD_DAYS)
    )

    period_record = calculate_period_activity(
        period_start=period_start,
        period_end=period_end,
        period_type="ROLLING_7_DAYS",
    )

    period_record["periodOffset"] = period_offset

    rolling_period_records.append(
        period_record
    )

rolling_weekly_activity_df = pd.DataFrame(
    rolling_period_records
)

rolling_weekly_activity_df = add_period_comparisons(
    rolling_weekly_activity_df
)


# ------------------------------------------------------------
# 07. Generate four-period historical benchmarks
# ------------------------------------------------------------

# Exclude the latest period from the historical benchmark.
#
# Including the current period in its own baseline would
# reduce the usefulness of anomaly detection.

current_period = (
    rolling_weekly_activity_df.iloc[-1]
)

historical_benchmark_periods_df = (
    rolling_weekly_activity_df.iloc[:-1].copy()
)

benchmark_metrics = {
    "pullRequestsOpened": {
        "metricId": "prsOpened",
        "unit": "count",
    },
    "pullRequestsMerged": {
        "metricId": "prsMerged",
        "unit": "count",
    },
    "pullRequestsClosed": {
        "metricId": "prsClosed",
        "unit": "count",
    },
    "medianMergeHours": {
        "metricId": "medianMergeHours",
        "unit": "hours",
    },
    "p90MergeHours": {
        "metricId": "p90MergeHours",
        "unit": "hours",
    },
}

weekly_benchmark_records = []

for source_column, metric_config in (
    benchmark_metrics.items()
):

    # Use only historical periods with estimated
    # boundary coverage and available metric values.

    benchmark_values = (
        historical_benchmark_periods_df.loc[
            historical_benchmark_periods_df[
                "windowBoundaryReached"
            ],
            source_column,
        ]
        .dropna()
    )

    current_value = current_period[source_column]

    previous_value = (
        rolling_weekly_activity_df.iloc[-2][
            source_column
        ]
    )

    benchmark_sample_size = len(
        benchmark_values
    )

    # Require all four preceding periods for the
    # configured four-period historical benchmark.

    benchmark_available = (
        benchmark_sample_size
        == BENCHMARK_PERIOD_COUNT
    )

    benchmark_mean = (
        benchmark_values.mean()
        if benchmark_available
        else float("nan")
    )

    benchmark_std = (
        benchmark_values.std(ddof=1)
        if benchmark_available
        else float("nan")
    )

    benchmark_change = (
        current_value - benchmark_mean
        if benchmark_available
        and current_period["windowBoundaryReached"]
        and pd.notna(current_value)
        else float("nan")
    )

    benchmark_change_pct = (
        benchmark_change / benchmark_mean * 100
        if pd.notna(benchmark_mean)
        and benchmark_mean > 0
        else float("nan")
    )

    # Standardized deviation is exploratory only.
    # Four observations are insufficient to establish
    # a robust statistical anomaly model.

    standardized_deviation = (
        benchmark_change / benchmark_std
        if pd.notna(benchmark_std)
        and benchmark_std > 0
        else float("nan")
    )

    weekly_benchmark_records.append(
        {
            "metricId": metric_config["metricId"],
            "unit": metric_config["unit"],
            "currentValue": current_value,
            "previousValue": previous_value,
            "historical4PeriodAvg": benchmark_mean,
            "historical4PeriodStd": benchmark_std,
            "changeVsBenchmark": benchmark_change,
            "changeVsBenchmarkPct": (
                benchmark_change_pct
            ),
            "standardizedDeviation": (
                standardized_deviation
            ),
            "benchmarkPeriodCount": (
                benchmark_sample_size
            ),
            "benchmarkAvailable": (
                benchmark_available
            ),
            "isCoverageGuaranteed": False,
        }
    )

weekly_benchmark_df = pd.DataFrame(
    weekly_benchmark_records
)


# ------------------------------------------------------------
# 08. Reconcile rolling periods with Cell 04
# ------------------------------------------------------------

# The latest rolling seven-day period must reconcile
# with the seven-day summary generated in Cell 04.
#
# The sum of the latest two non-overlapping periods
# must reconcile with the fourteen-day summary for
# additive metrics only.
#
# Medians and percentiles are not additive.

volume_lookup = (
    historical_volume_summary_df
    .set_index("windowDays")
)

additive_metrics = [
    "pullRequestsOpened",
    "pullRequestsMerged",
    "pullRequestsClosed",
    "closedWithoutMerge",
]

weekly_validation_records = []

for metric in additive_metrics:

    current_7_day_value = int(
        rolling_weekly_activity_df.iloc[-1][metric]
    )

    expected_7_day_value = int(
        volume_lookup.loc[7, metric]
    )

    current_14_day_value = int(
        rolling_weekly_activity_df.tail(2)[metric].sum()
    )

    expected_14_day_value = int(
        volume_lookup.loc[14, metric]
    )

    for window_days, actual, expected in [
        (7, current_7_day_value, expected_7_day_value),
        (14, current_14_day_value, expected_14_day_value),
    ]:

        weekly_validation_records.append(
            {
                "validationRule": (
                    f"{window_days}-day {metric} reconciliation"
                ),
                "actualValue": actual,
                "expectedValue": expected,
                "difference": actual - expected,
                "validationStatus": (
                    "PASS"
                    if actual == expected
                    else "FAIL"
                ),
            }
        )

weekly_validation_df = pd.DataFrame(
    weekly_validation_records
)

all_weekly_checks_passed = bool(
    weekly_validation_df["validationStatus"]
    .eq("PASS")
    .all()
)


# ------------------------------------------------------------
# 09. Display calendar week activity
# ------------------------------------------------------------

calendar_display_df = weekly_activity_df[
    [
        "periodStart",
        "periodEndExclusive",
        "coverageStatus",
        "pullRequestsOpened",
        "pullRequestsMerged",
        "pullRequestsClosed",
        "medianMergeHours",
        "p90MergeHours",
        "mergedWoWPct",
    ]
].copy()

# Format dates for presentation without changing the
# underlying UTC timestamp columns.

calendar_display_df["periodStart"] = (
    calendar_display_df["periodStart"].dt.date
)

calendar_display_df["periodEndExclusive"] = (
    calendar_display_df["periodEndExclusive"].dt.date
)

print("EIP | Calendar Weekly Activity")
print("=" * 75)

display(
    calendar_display_df.round(
        {
            "medianMergeHours": 2,
            "p90MergeHours": 2,
            "mergedWoWPct": 2,
        }
    )
)


# ------------------------------------------------------------
# 10. Display rolling seven-day activity
# ------------------------------------------------------------

rolling_display_df = rolling_weekly_activity_df[
    [
        "periodOffset",
        "periodStart",
        "periodEndExclusive",
        "coverageStatus",
        "pullRequestsOpened",
        "pullRequestsMerged",
        "pullRequestsClosed",
        "olderPullRequestsMerged",
        "medianMergeHours",
        "meanMergeHours",
        "p90MergeHours",
        "mergeDurationSampleSize",
    ]
].copy()

rolling_display_df["periodStart"] = (
    rolling_display_df["periodStart"].dt.date
)

rolling_display_df["periodEndExclusive"] = (
    rolling_display_df["periodEndExclusive"].dt.date
)

print("\nEIP | Rolling Seven-Day Activity")
print("=" * 75)

display(
    rolling_display_df.round(
        {
            "medianMergeHours": 2,
            "meanMergeHours": 2,
            "p90MergeHours": 2,
        }
    )
)


# ------------------------------------------------------------
# 11. Display latest period-over-period comparison
# ------------------------------------------------------------

latest_comparison_df = (
    rolling_weekly_activity_df.tail(1)[
        [
            "periodStart",
            "periodEndExclusive",
            "openedWoWPct",
            "mergedWoWPct",
            "closedWoWPct",
            "medianMergeHoursWoWPct",
        ]
    ]
)

print("\nEIP | Latest Period-over-Period Comparison")
print("=" * 75)

display(
    latest_comparison_df.round(
        {
            "openedWoWPct": 2,
            "mergedWoWPct": 2,
            "closedWoWPct": 2,
            "medianMergeHoursWoWPct": 2,
        }
    )
)


# ------------------------------------------------------------
# 12. Display historical benchmark results
# ------------------------------------------------------------

print("\nEIP | Four-Period Historical Benchmarks")
print("=" * 75)

display(
    weekly_benchmark_df.round(2)
)


# ------------------------------------------------------------
# 13. Display reconciliation results
# ------------------------------------------------------------

print("\nEIP | Weekly Activity Reconciliation")
print("=" * 75)

display(weekly_validation_df)

print(
    "All Weekly Reconciliation Checks Passed:",
    all_weekly_checks_passed,
)


# ------------------------------------------------------------
# 14. Generate final execution summary
# ------------------------------------------------------------

print("\nEIP | FINAL WEEKLY PROFILING SUMMARY")
print("=" * 75)

print(
    "Calendar Reporting Weeks:",
    len(weekly_activity_df),
)

print(
    "Rolling Seven-Day Periods:",
    len(rolling_weekly_activity_df),
)

print(
    "Historical Benchmark Periods:",
    len(historical_benchmark_periods_df),
)

print(
    "Benchmark Metrics Generated:",
    len(weekly_benchmark_df),
)

print(
    "Available Historical Benchmarks:",
    int(
        weekly_benchmark_df[
            "benchmarkAvailable"
        ].sum()
    ),
)

print(
    "Validation Rules Executed:",
    len(weekly_validation_df),
)

print(
    "Validation Rules Passed:",
    int(
        weekly_validation_df[
            "validationStatus"
        ].eq("PASS").sum()
    ),
)

print("Additional API Requests Executed: 0")
print("Guaranteed Historical Coverage: False")
print("Production Anomaly Detection Implemented: False")
print("Database Persistence Completed: False")

print(
    "\nWeekly Profiling Status:",
    "COMPLETED_WITH_LIMITATIONS"
    if all_weekly_checks_passed
    else "COMPLETED_WITH_VALIDATION_FAILURES",
)

EIP | Calendar Weekly Activity


,periodStart,periodEndExclusive,coverageStatus,pullRequestsOpened,pullRequestsMerged,pullRequestsClosed,medianMergeHours,p90MergeHours,mergedWoWPct
0,2026-09-28,2026-10-05,ESTIMATED_COVERAGE,338,252,391,49.56,969.93,NaN
1,2026-10-05,2026-10-12,ESTIMATED_COVERAGE,311,239,305,48.11,508.30,-5.16
2,2026-10-12,2026-10-19,ESTIMATED_COVERAGE,409,324,375,19.58,1191.95,35.56
3,2026-10-19,2026-10-26,ESTIMATED_COVERAGE,562,414,493,9.63,278.14,27.78
4,2026-10-26,2026-11-02,ESTIMATED_WITH_QUARANTINE,593,482,565,6.37,258.36,16.43
5,2026-11-02,2026-11-09,ESTIMATED_COVERAGE,402,329,457,24.24,710.97,-31.74
6,2026-11-09,2026-11-16,ESTIMATED_COVERAGE,369,242,294,25.33,400.24,-26.44
7,2026-11-16,2026-11-23,ESTIMATED_COVERAGE,298,200,257,59.91,558.93,-17.36



EIP | Rolling Seven-Day Activity


,periodOffset,periodStart,periodEndExclusive,coverageStatus,pullRequestsOpened,pullRequestsMerged,pullRequestsClosed,olderPullRequestsMerged,medianMergeHours,meanMergeHours,p90MergeHours,mergeDurationSampleSize
0,4,2026-10-22,2026-10-29,ESTIMATED_COVERAGE,519,404,469,99,6.56,115.93,212.61,404
1,3,2026-10-29,2026-11-05,ESTIMATED_WITH_QUARANTINE,539,416,546,111,9.62,148.76,308.03,416
2,2,2026-11-05,2026-11-12,ESTIMATED_COVERAGE,396,302,366,142,43.71,265.99,673.70,302
3,1,2026-11-12,2026-11-19,ESTIMATED_COVERAGE,273,222,270,115,49.67,267.82,485.48,222
4,0,2026-11-19,2026-11-26,ESTIMATED_COVERAGE,305,208,300,89,66.53,252.09,532.73,208



EIP | Latest Period-over-Period Comparison


,periodStart,periodEndExclusive,openedWoWPct,mergedWoWPct,closedWoWPct,medianMergeHoursWoWPct
4,2026-11-19 00:00:00+00:00,2026-11-26 00:00:00+00:00,11.72,-6.31,11.11,33.94



EIP | Four-Period Historical Benchmarks


,metricId,unit,currentValue,previousValue,historical4PeriodAvg,historical4PeriodStd,changeVsBenchmark,changeVsBenchmarkPct,standardizedDeviation,benchmarkPeriodCount,benchmarkAvailable,isCoverageGuaranteed
0,prsOpened,count,305.00,273.00,431.75,123.28,-126.75,-29.36,-1.03,4,True,False
1,prsMerged,count,208.00,222.00,336.00,91.61,-128.00,-38.10,-1.40,4,True,False
2,prsClosed,count,300.00,270.00,412.75,120.39,-112.75,-27.32,-0.94,4,True,False
3,medianMergeHours,hours,66.53,49.67,27.39,22.45,39.14,142.89,1.74,4,True,False
4,p90MergeHours,hours,532.73,485.48,419.95,203.47,112.77,26.85,0.55,4,True,False



EIP | Weekly Activity Reconciliation


,validationRule,actualValue,expectedValue,difference,validationStatus
0,7-day pullRequestsOpened reconciliation,305,305,0,PASS
1,14-day pullRequestsOpened reconciliation,578,578,0,PASS
2,7-day pullRequestsMerged reconciliation,208,208,0,PASS
3,14-day pullRequestsMerged reconciliation,430,430,0,PASS
4,7-day pullRequestsClosed reconciliation,300,300,0,PASS
5,14-day pullRequestsClosed reconciliation,570,570,0,PASS
6,7-day closedWithoutMerge reconciliation,92,92,0,PASS
7,14-day closedWithoutMerge reconciliation,140,140,0,PASS


All Weekly Reconciliation Checks Passed: True

EIP | FINAL WEEKLY PROFILING SUMMARY
Calendar Reporting Weeks: 8
Rolling Seven-Day Periods: 5
Historical Benchmark Periods: 4
Benchmark Metrics Generated: 5
Available Historical Benchmarks: 5
Validation Rules Executed: 8
Validation Rules Passed: 8
Additional API Requests Executed: 0
Guaranteed Historical Coverage: False
Production Anomaly Detection Implemented: False
Database Persistence Completed: False

Weekly Profiling Status: COMPLETED_WITH_LIMITATIONS


In [8]:
# ============================================================
# Engineering Intelligence Platform (EIP)
# Section 05: API Consumption and Ingestion Feasibility
#
# Purpose:
# Evaluate alternative source enrichment strategies,
# estimate API consumption and determine the initial
# production ingestion scope.
#
# Dependencies:
# Section 02A: Historical Pull Request Extraction
# Section 02B: Historical Source Validation
# Section 03: Historical Activity and Volume Profiling
# Section 04: Weekly Activity and Historical Benchmarks
#
# Outputs:
# - ingestion_strategy_summary_df
# - recommended_ingestion_config_df
# - metric_availability_summary_df
# - ingestion_feasibility_validation_df
#
# Design Principles:
# - Separate basic ingestion from detailed enrichment.
# - Enrich complete observed analytical cohorts.
# - Preserve source coverage metadata.
# - Maintain API quota reserves.
# - Avoid unnecessary upstream requests.
#
# Note:
# All request estimates are based on observed source
# volumes. Actual review pagination and retries may vary.
#
# No additional API requests are executed in this cell.
# ============================================================


# ------------------------------------------------------------
# 01. Import additional dependencies
# ------------------------------------------------------------

import math


# ------------------------------------------------------------
# 02. Validate profiling dependencies
# ------------------------------------------------------------

required_variables = [
    "historical_volume_summary_df",
    "historical_enrichment_summary_df",
    "historical_coverage_assessment_df",
    "historical_pull_requests_df",
    "historical_window_datasets",
    "rolling_weekly_activity_df",
    "weekly_benchmark_df",
    "pagination_summary_df",
    "remaining_api_requests",
    "AS_OF_TIMESTAMP",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:

    raise RuntimeError(
        f"Missing profiling dependencies: {missing_variables}"
    )


# ------------------------------------------------------------
# 03. Configure ingestion planning parameters
# ------------------------------------------------------------

# Proposed historical coverage for basic PR ingestion.

BASIC_HISTORY_DAYS = 60

# Proposed detailed enrichment coverage.
# Two non-overlapping seven-day reporting periods.

ENRICHMENT_HISTORY_DAYS = 14

# Original and revised demonstration request budgets.

ORIGINAL_DEMO_REQUEST_BUDGET = 1000

PROPOSED_DEMO_REQUEST_BUDGET = 1200

# Preserve a minimum API quota reserve.

API_RESERVE_REQUESTS = MIN_REMAINING_REQUESTS

# Allow additional requests for review pagination,
# retries and other extraction variability.
#
# This is a planning allowance, not a guaranteed bound.

REQUEST_CONTINGENCY_PCT = 0.20

REPOSITORY_METADATA_REQUESTS = 1

DETAIL_REQUESTS_PER_PR = 1

BASE_REVIEW_REQUESTS_PER_PR = 1


# ------------------------------------------------------------
# 04. Establish available API request budget
# ------------------------------------------------------------

# Use the latest observed primary quota from exploration.
# The actual available quota must be checked again before
# executing the production ingestion pipeline.

available_api_budget = max(
    0,
    remaining_api_requests - API_RESERVE_REQUESTS,
)

effective_demo_budget = min(
    available_api_budget,
    PROPOSED_DEMO_REQUEST_BUDGET,
)

print("EIP | API Budget Configuration")
print("=" * 75)

print(
    "Last Observed Remaining API Requests:",
    remaining_api_requests,
)

print(
    "Reserved API Requests:",
    API_RESERVE_REQUESTS,
)

print(
    "Available API Budget:",
    available_api_budget,
)

print(
    "Original Demo Budget:",
    ORIGINAL_DEMO_REQUEST_BUDGET,
)

print(
    "Proposed Demo Budget:",
    PROPOSED_DEMO_REQUEST_BUDGET,
)

print(
    "Effective Demo Budget:",
    effective_demo_budget,
)

print(
    "Request Contingency Percentage:",
    REQUEST_CONTINGENCY_PCT * 100,
)


# ------------------------------------------------------------
# 05. Prepare historical profiling lookups
# ------------------------------------------------------------

volume_lookup = (
    historical_volume_summary_df
    .set_index("windowDays")
)

enrichment_lookup = (
    historical_enrichment_summary_df
    .set_index("windowDays")
)

coverage_lookup = (
    historical_coverage_assessment_df
    .set_index("windowDays")
)

basic_history_candidates = int(
    volume_lookup.loc[
        BASIC_HISTORY_DAYS,
        "candidatePullRequests",
    ]
)

basic_history_boundary_reached = bool(
    coverage_lookup.loc[
        BASIC_HISTORY_DAYS,
        "windowBoundaryReached",
    ]
)

# Use the observed page on which the historical boundary
# was confirmed, rather than estimating list requests
# simply by dividing candidate count by page size.

basic_boundary_page = coverage_lookup.loc[
    BASIC_HISTORY_DAYS,
    "boundaryConfirmationPage",
]

basic_list_requests = (
    int(basic_boundary_page)
    if pd.notna(basic_boundary_page)
    else len(pagination_summary_df)
)

basic_source_requests = (
    REPOSITORY_METADATA_REQUESTS
    + basic_list_requests
)


# ------------------------------------------------------------
# 06. Define alternative ingestion strategies
# ------------------------------------------------------------

ingestion_strategies = [
    {
        "strategyId": "STR-001",
        "strategyName": "Basic PR ingestion only",
        "enrichmentDays": 0,
        "detailPRs": 0,
        "reviewPRs": 0,
    },
    {
        "strategyId": "STR-002",
        "strategyName": "Full 60-day enrichment",
        "enrichmentDays": 60,
        "detailPRs": basic_history_candidates,
        "reviewPRs": basic_history_candidates,
    },
    {
        "strategyId": "STR-003",
        "strategyName": "7-day merged cohort enrichment",
        "enrichmentDays": 7,
        "detailPRs": int(
            enrichment_lookup.loc[
                7,
                "mergedCohortPullRequests",
            ]
        ),
        "reviewPRs": int(
            enrichment_lookup.loc[
                7,
                "mergedCohortPullRequests",
            ]
        ),
    },
    {
        "strategyId": "STR-004",
        "strategyName": "14-day merged cohort reviews only",
        "enrichmentDays": 14,
        "detailPRs": 0,
        "reviewPRs": int(
            enrichment_lookup.loc[
                14,
                "mergedCohortPullRequests",
            ]
        ),
    },
    {
        "strategyId": "STR-005",
        "strategyName": "14-day merged cohort enrichment",
        "enrichmentDays": 14,
        "detailPRs": int(
            enrichment_lookup.loc[
                14,
                "mergedCohortPullRequests",
            ]
        ),
        "reviewPRs": int(
            enrichment_lookup.loc[
                14,
                "mergedCohortPullRequests",
            ]
        ),
    },
    {
        "strategyId": "STR-006",
        "strategyName": "30-day merged cohort enrichment",
        "enrichmentDays": 30,
        "detailPRs": int(
            enrichment_lookup.loc[
                30,
                "mergedCohortPullRequests",
            ]
        ),
        "reviewPRs": int(
            enrichment_lookup.loc[
                30,
                "mergedCohortPullRequests",
            ]
        ),
    },
]


# ------------------------------------------------------------
# 07. Estimate API consumption by ingestion strategy
# ------------------------------------------------------------

ingestion_strategy_records = []

for strategy in ingestion_strategies:

    enrichment_days = strategy["enrichmentDays"]

    detail_requests = (
        strategy["detailPRs"]
        * DETAIL_REQUESTS_PER_PR
    )

    review_requests = (
        strategy["reviewPRs"]
        * BASE_REVIEW_REQUESTS_PER_PR
    )

    baseline_requests = (
        basic_source_requests
        + detail_requests
        + review_requests
    )

    contingency_requests = math.ceil(
        baseline_requests
        * REQUEST_CONTINGENCY_PCT
    )

    estimated_requests = (
        baseline_requests
        + contingency_requests
    )

    enrichment_boundary_reached = (
        bool(
            coverage_lookup.loc[
                enrichment_days,
                "windowBoundaryReached",
            ]
        )
        if enrichment_days > 0
        else True
    )

    estimated_boundary_coverage = (
        basic_history_boundary_reached
        and enrichment_boundary_reached
    )

    fits_original_budget = (
        estimated_boundary_coverage
        and estimated_requests
        <= min(
            available_api_budget,
            ORIGINAL_DEMO_REQUEST_BUDGET,
        )
    )

    fits_proposed_budget = (
        estimated_boundary_coverage
        and estimated_requests <= effective_demo_budget
    )

    if not estimated_boundary_coverage:

        feasibility_status = "INCOMPLETE_BOUNDARY_COVERAGE"

    elif estimated_requests > available_api_budget:

        feasibility_status = "EXCEEDS_AVAILABLE_API_BUDGET"

    elif not fits_proposed_budget:

        feasibility_status = "EXCEEDS_DEMO_BUDGET"

    else:

        feasibility_status = "WITHIN_DEMO_BUDGET"

    ingestion_strategy_records.append(
        {
            **strategy,

            "basicHistoryDays": BASIC_HISTORY_DAYS,

            "basicCandidatePRs": basic_history_candidates,

            "repositoryRequests": (
                REPOSITORY_METADATA_REQUESTS
            ),

            "listRequests": basic_list_requests,

            "detailRequests": detail_requests,

            "minimumReviewRequests": review_requests,

            "baselineRequests": baseline_requests,

            "contingencyRequests": contingency_requests,

            "estimatedTotalRequests": estimated_requests,

            "fitsOriginalDemoBudget": (
                fits_original_budget
            ),

            "fitsProposedDemoBudget": (
                fits_proposed_budget
            ),

            "estimatedBoundaryCoverage": (
                estimated_boundary_coverage
            ),

            "feasibilityStatus": feasibility_status,
        }
    )

ingestion_strategy_summary_df = pd.DataFrame(
    ingestion_strategy_records
)


# ------------------------------------------------------------
# 08. Select the recommended ingestion strategy
# ------------------------------------------------------------

# Prefer comparable detail and review enrichment across
# two complete rolling seven-day reporting periods.
#
# If the proposed request budget is insufficient,
# fall back to review-only enrichment for both periods.
#
# The final fallback provides basic lifecycle metrics.

preferred_strategy_order = [
    "STR-005",
    "STR-004",
    "STR-001",
]

recommended_strategy = None

for strategy_id in preferred_strategy_order:

    eligible_strategy_df = (
        ingestion_strategy_summary_df.loc[
            ingestion_strategy_summary_df[
                "strategyId"
            ].eq(strategy_id)

            & ingestion_strategy_summary_df[
                "fitsProposedDemoBudget"
            ]
        ]
    )

    if not eligible_strategy_df.empty:

        recommended_strategy = (
            eligible_strategy_df.iloc[0]
        )

        break

if recommended_strategy is None:

    raise RuntimeError(
        "No proposed ingestion strategy satisfies "
        "the configured request budget and coverage rules."
    )

selected_strategy_id = (
    recommended_strategy["strategyId"]
)

selected_enrichment_days = int(
    recommended_strategy["enrichmentDays"]
)

selected_detail_count = int(
    recommended_strategy["detailPRs"]
)

selected_review_count = int(
    recommended_strategy["reviewPRs"]
)


# ------------------------------------------------------------
# 09. Build proposed enrichment cohort
# ------------------------------------------------------------

# Select PRs by merge timestamp rather than creation
# timestamp or current update activity.
#
# This preserves older PRs that were merged during the
# selected reporting periods.

enrichment_start = (
    AS_OF_TIMESTAMP
    - pd.Timedelta(days=selected_enrichment_days)
)

if selected_enrichment_days > 0:

    enrichment_cohort_mask = within_reporting_window(
        historical_pull_requests_df["merged_at"],
        enrichment_start,
        AS_OF_TIMESTAMP,
    )

    recommended_enrichment_cohort_df = (
        historical_pull_requests_df.loc[
            enrichment_cohort_mask
        ]
        .copy()
    )

else:

    recommended_enrichment_cohort_df = (
        historical_pull_requests_df.iloc[0:0]
        .copy()
    )


# ------------------------------------------------------------
# 10. Assign enrichment reporting periods
# ------------------------------------------------------------

# Separate the current and previous seven-day cohorts
# to support comparable collaboration metrics.

latest_period_start = (
    AS_OF_TIMESTAMP
    - pd.Timedelta(days=7)
)

recommended_enrichment_cohort_df[
    "reportingPeriod"
] = "EARLIER_PERIOD"

recommended_enrichment_cohort_df.loc[
    recommended_enrichment_cohort_df[
        "merged_at"
    ].ge(latest_period_start),

    "reportingPeriod",
] = "CURRENT_PERIOD"

previous_period_start = (
    AS_OF_TIMESTAMP
    - pd.Timedelta(days=14)
)

previous_period_mask = (
    within_reporting_window(
        recommended_enrichment_cohort_df["merged_at"],
        previous_period_start,
        latest_period_start,
    )
)

recommended_enrichment_cohort_df.loc[
    previous_period_mask,
    "reportingPeriod",
] = "PREVIOUS_PERIOD"

# These flags represent planned extraction requirements,
# not completed enrichment.

recommended_enrichment_cohort_df[
    "requiresDetailExtraction"
] = selected_detail_count > 0

recommended_enrichment_cohort_df[
    "requiresReviewExtraction"
] = selected_review_count > 0

enrichment_period_summary_df = (
    recommended_enrichment_cohort_df
    .groupby(
        "reportingPeriod",
        as_index=False,
    )
    .agg(
        pullRequestCount=("id", "nunique"),
        plannedDetailRequests=(
            "requiresDetailExtraction",
            "sum",
        ),
        minimumReviewRequests=(
            "requiresReviewExtraction",
            "sum",
        ),
    )
)


# ------------------------------------------------------------
# 11. Build proposed production configuration
# ------------------------------------------------------------

# This is a configuration prototype for the future
# Control layer. No database objects are created here.

recommended_ingestion_config_df = pd.DataFrame(
    [
        {
            "configKey": "basicHistoryDays",
            "configValue": BASIC_HISTORY_DAYS,
            "description": (
                "Historical lookback for basic PR ingestion"
            ),
        },
        {
            "configKey": "enrichmentHistoryDays",
            "configValue": selected_enrichment_days,
            "description": (
                "Historical lookback for merged cohort enrichment"
            ),
        },
        {
            "configKey": "enrichmentCohort",
            "configValue": "MERGED_PULL_REQUESTS",
            "description": (
                "PRs merged within the enrichment period"
            ),
        },
        {
            "configKey": "reportingPeriodDays",
            "configValue": 7,
            "description": (
                "Default comparison period length"
            ),
        },
        {
            "configKey": "benchmarkPeriodCount",
            "configValue": 4,
            "description": (
                "Historical periods used for basic metric benchmarks"
            ),
        },
        {
            "configKey": "maxListPages",
            "configValue": MAX_PAGES,
            "description": (
                "Initial bounded PR list pagination limit"
            ),
        },
        {
            "configKey": "recordsPerPage",
            "configValue": PAGE_SIZE,
            "description": (
                "Requested source records per API page"
            ),
        },
        {
            "configKey": "maxApiRequestsPerRun",
            "configValue": PROPOSED_DEMO_REQUEST_BUDGET,
            "description": (
                "Proposed hard request cap per ingestion execution"
            ),
        },
        {
            "configKey": "minimumApiReserve",
            "configValue": API_RESERVE_REQUESTS,
            "description": (
                "Minimum primary API quota reserve"
            ),
        },
        {
            "configKey": "requestContingencyPct",
            "configValue": REQUEST_CONTINGENCY_PCT * 100,
            "description": (
                "Planning allowance for additional API requests"
            ),
        },
        {
            "configKey": "incrementalOverlapHours",
            "configValue": 48,
            "description": (
                "Proposed overlapping incremental extraction window"
            ),
        },
        {
            "configKey": "reportingTimezone",
            "configValue": "UTC",
            "description": (
                "Standard timezone for analytical calculations"
            ),
        },
    ]
)


# ------------------------------------------------------------
# 12. Define metric availability by source coverage
# ------------------------------------------------------------

# Basic lifecycle metrics require only PR list records.
#
# Detailed collaboration metrics require completed review
# extraction for every eligible PR in their stated cohort.
#
# Change-size metrics require PR detail extraction.

review_metric_status = (
    "PLANNED_ENRICHMENT"
    if selected_review_count > 0
    else "NOT_INCLUDED"
)

detail_metric_status = (
    "PLANNED_ENRICHMENT"
    if selected_detail_count > 0
    else "NOT_INCLUDED"
)

metric_availability_records = [
    (
        "prsOpened",
        "SRC-002",
        BASIC_HISTORY_DAYS,
        "CREATED_IN_PERIOD",
        "OBSERVED_SOURCE_AVAILABLE",
    ),
    (
        "prsMerged",
        "SRC-002",
        BASIC_HISTORY_DAYS,
        "MERGED_IN_PERIOD",
        "OBSERVED_SOURCE_AVAILABLE",
    ),
    (
        "prsClosedWithoutMerge",
        "SRC-002",
        BASIC_HISTORY_DAYS,
        "CLOSED_WITHOUT_MERGE_IN_PERIOD",
        "OBSERVED_SOURCE_AVAILABLE",
    ),
    (
        "medianMergeHours",
        "SRC-002",
        BASIC_HISTORY_DAYS,
        "MERGED_IN_PERIOD",
        "OBSERVED_SOURCE_AVAILABLE",
    ),
    (
        "p90MergeHours",
        "SRC-002",
        BASIC_HISTORY_DAYS,
        "MERGED_IN_PERIOD",
        "OBSERVED_SOURCE_AVAILABLE",
    ),
    (
        "medianFirstReviewHours",
        "SRC-002, SRC-004",
        selected_enrichment_days,
        "REVIEWED_MERGED_IN_PERIOD",
        review_metric_status,
    ),
    (
        "meanPreReviewHours",
        "SRC-002, SRC-004",
        selected_enrichment_days,
        "REVIEWED_MERGED_IN_PERIOD",
        review_metric_status,
    ),
    (
        "meanPostReviewHours",
        "SRC-002, SRC-004",
        selected_enrichment_days,
        "REVIEWED_MERGED_IN_PERIOD",
        review_metric_status,
    ),
    (
        "preReviewTimeSharePct",
        "SRC-002, SRC-004",
        selected_enrichment_days,
        "REVIEWED_MERGED_IN_PERIOD",
        review_metric_status,
    ),
    (
        "mergedPRReviewCoveragePct",
        "SRC-002, SRC-004",
        selected_enrichment_days,
        "MERGED_IN_PERIOD",
        review_metric_status,
    ),
    (
        "uniqueExternalReviewers",
        "SRC-002, SRC-004",
        selected_enrichment_days,
        "MERGED_IN_PERIOD",
        review_metric_status,
    ),
    (
        "medianLinesChanged",
        "SRC-002, SRC-003",
        selected_enrichment_days,
        "MERGED_IN_PERIOD",
        detail_metric_status,
    ),
    (
        "medianFilesChanged",
        "SRC-002, SRC-003",
        selected_enrichment_days,
        "MERGED_IN_PERIOD",
        detail_metric_status,
    ),
]

metric_availability_summary_df = pd.DataFrame(
    metric_availability_records,
    columns=[
        "metricId",
        "sourceDependencies",
        "plannedHistoryDays",
        "analyticalCohort",
        "availabilityStatus",
    ],
)

# Detailed historical benchmarks require additional
# enrichment periods and are not part of the initial MVP.

metric_availability_summary_df[
    "fourPeriodBenchmarkAvailable"
] = (
    metric_availability_summary_df[
        "metricId"
    ].isin(
        weekly_benchmark_df.loc[
            weekly_benchmark_df[
                "benchmarkAvailable"
            ],
            "metricId",
        ]
    )
)


# ------------------------------------------------------------
# 13. Validate ingestion feasibility
# ------------------------------------------------------------

# Validate the proposed enrichment cohort against
# the actual merged PR population.

expected_enrichment_count = max(
    selected_detail_count,
    selected_review_count,
)

actual_enrichment_count = len(
    recommended_enrichment_cohort_df
)

estimated_total_requests = int(
    recommended_strategy["estimatedTotalRequests"]
)

ingestion_validation_results = {
    "Basic historical boundary reached": (
        basic_history_boundary_reached
    ),

    "Selected enrichment cohort reconciled": (
        actual_enrichment_count
        == expected_enrichment_count
    ),

    "Enrichment identifiers unique": (
        not recommended_enrichment_cohort_df[
            "id"
        ].duplicated().any()
    ),

    "Proposed request budget respected": (
        estimated_total_requests
        <= effective_demo_budget
    ),

    "Primary API reserve preserved in estimate": (
        estimated_total_requests
        <= available_api_budget
    ),

    "Basic historical benchmark window sufficient": (
        BASIC_HISTORY_DAYS
        >= REPORTING_PERIOD_DAYS
        * (BENCHMARK_PERIOD_COUNT + 1)
    ),
}

ingestion_feasibility_validation_df = pd.DataFrame(
    [
        {
            "validationRule": rule,
            "validationStatus": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
        for rule, passed
        in ingestion_validation_results.items()
    ]
)

all_ingestion_checks_passed = all(
    ingestion_validation_results.values()
)


# ------------------------------------------------------------
# 14. Display alternative ingestion strategies
# ------------------------------------------------------------

print("\nEIP | Ingestion Strategy Comparison")
print("=" * 75)

display(
    ingestion_strategy_summary_df[
        [
            "strategyId",
            "strategyName",
            "basicHistoryDays",
            "enrichmentDays",
            "detailRequests",
            "minimumReviewRequests",
            "baselineRequests",
            "contingencyRequests",
            "estimatedTotalRequests",
            "feasibilityStatus",
        ]
    ]
)


# ------------------------------------------------------------
# 15. Display selected enrichment cohorts
# ------------------------------------------------------------

print("\nEIP | Selected Enrichment Cohorts")
print("=" * 75)

display(enrichment_period_summary_df)


# ------------------------------------------------------------
# 16. Display proposed production configuration
# ------------------------------------------------------------

print("\nEIP | Recommended Ingestion Configuration")
print("=" * 75)

display(recommended_ingestion_config_df)


# ------------------------------------------------------------
# 17. Display metric availability
# ------------------------------------------------------------

print("\nEIP | Proposed Metric Availability")
print("=" * 75)

display(metric_availability_summary_df)


# ------------------------------------------------------------
# 18. Display validation results
# ------------------------------------------------------------

print("\nEIP | Ingestion Feasibility Validation")
print("=" * 75)

display(ingestion_feasibility_validation_df)


# ------------------------------------------------------------
# 19. Generate final ingestion recommendation
# ------------------------------------------------------------

print("\nEIP | FINAL INGESTION RECOMMENDATION")
print("=" * 75)

print(
    "Selected Strategy:",
    recommended_strategy["strategyName"],
)

print(
    "Basic Historical Coverage:",
    BASIC_HISTORY_DAYS,
    "days",
)

print(
    "Observed Basic PR Candidates:",
    basic_history_candidates,
)

print(
    "Detailed Enrichment Coverage:",
    selected_enrichment_days,
    "days",
)

print(
    "Selected Enrichment PRs:",
    actual_enrichment_count,
)

print(
    "Planned Repository Requests:",
    REPOSITORY_METADATA_REQUESTS,
)

print(
    "Planned PR List Requests:",
    basic_list_requests,
)

print(
    "Planned PR Detail Requests:",
    selected_detail_count,
)

print(
    "Minimum PR Review Requests:",
    selected_review_count,
)

print(
    "Baseline API Requests:",
    int(
        recommended_strategy["baselineRequests"]
    ),
)

print(
    "Contingency Requests:",
    int(
        recommended_strategy["contingencyRequests"]
    ),
)

print(
    "Estimated Total API Requests:",
    estimated_total_requests,
)

print(
    "Effective Demo Budget:",
    effective_demo_budget,
)

print(
    "Estimated Budget Headroom:",
    effective_demo_budget
    - estimated_total_requests,
)

print(
    "All Ingestion Feasibility Checks Passed:",
    all_ingestion_checks_passed,
)

print("Additional API Requests Executed: 0")
print("Guaranteed Historical Coverage: False")
print("Historical Details Extracted: False")
print("Historical Reviews Extracted: False")
print("Production Ingestion Implemented: False")
print("Database Persistence Completed: False")

print(
    "\nIngestion Planning Status:",
    "COMPLETED_WITH_LIMITATIONS"
    if all_ingestion_checks_passed
    else "COMPLETED_WITH_VALIDATION_FAILURES",
)

EIP | API Budget Configuration
Last Observed Remaining API Requests: 4850
Reserved API Requests: 500
Available API Budget: 4350
Original Demo Budget: 1000
Proposed Demo Budget: 1200
Effective Demo Budget: 1200
Request Contingency Percentage: 20.0

EIP | Ingestion Strategy Comparison


,strategyId,strategyName,basicHistoryDays,enrichmentDays,detailRequests,minimumReviewRequests,baselineRequests,contingencyRequests,estimatedTotalRequests,feasibilityStatus
0,STR-001,Basic PR ingestion only,60,0,0,0,51,11,62,WITHIN_DEMO_BUDGET
1,STR-002,Full 60-day enrichment,60,60,4707,4707,9465,1893,11358,EXCEEDS_AVAILABLE_API_BUDGET
2,STR-003,7-day merged cohort enrichment,60,7,208,208,467,94,561,WITHIN_DEMO_BUDGET
3,STR-004,14-day merged cohort reviews only,60,14,0,430,481,97,578,WITHIN_DEMO_BUDGET
4,STR-005,14-day merged cohort enrichment,60,14,430,430,911,183,1094,WITHIN_DEMO_BUDGET
5,STR-006,30-day merged cohort enrichment,60,30,1305,1305,2661,533,3194,EXCEEDS_DEMO_BUDGET



EIP | Selected Enrichment Cohorts


,reportingPeriod,pullRequestCount,plannedDetailRequests,minimumReviewRequests
0,CURRENT_PERIOD,208,208,208
1,PREVIOUS_PERIOD,222,222,222



EIP | Recommended Ingestion Configuration


,configKey,configValue,description
0,basicHistoryDays,60,Historical lookback for basic PR ingestion
1,enrichmentHistoryDays,14,Historical lookback for merged cohort enrichment
2,enrichmentCohort,MERGED_PULL_REQUESTS,PRs merged within the enrichment period
3,reportingPeriodDays,7,Default comparison period length
4,benchmarkPeriodCount,4,Historical periods used for basic metric benchmarks
5,maxListPages,50,Initial bounded PR list pagination limit
6,recordsPerPage,100,Requested source records per API page
7,maxApiRequestsPerRun,1200,Proposed hard request cap per ingestion execution
8,minimumApiReserve,500,Minimum primary API quota reserve
9,requestContingencyPct,20.0,Planning allowance for additional API requests



EIP | Proposed Metric Availability


,metricId,sourceDependencies,plannedHistoryDays,analyticalCohort,availabilityStatus,fourPeriodBenchmarkAvailable
0,prsOpened,SRC-002,60,CREATED_IN_PERIOD,OBSERVED_SOURCE_AVAILABLE,True
1,prsMerged,SRC-002,60,MERGED_IN_PERIOD,OBSERVED_SOURCE_AVAILABLE,True
2,prsClosedWithoutMerge,SRC-002,60,CLOSED_WITHOUT_MERGE_IN_PERIOD,OBSERVED_SOURCE_AVAILABLE,False
3,medianMergeHours,SRC-002,60,MERGED_IN_PERIOD,OBSERVED_SOURCE_AVAILABLE,True
4,p90MergeHours,SRC-002,60,MERGED_IN_PERIOD,OBSERVED_SOURCE_AVAILABLE,True
5,medianFirstReviewHours,"SRC-002, SRC-004",14,REVIEWED_MERGED_IN_PERIOD,PLANNED_ENRICHMENT,False
6,meanPreReviewHours,"SRC-002, SRC-004",14,REVIEWED_MERGED_IN_PERIOD,PLANNED_ENRICHMENT,False
7,meanPostReviewHours,"SRC-002, SRC-004",14,REVIEWED_MERGED_IN_PERIOD,PLANNED_ENRICHMENT,False
8,preReviewTimeSharePct,"SRC-002, SRC-004",14,REVIEWED_MERGED_IN_PERIOD,PLANNED_ENRICHMENT,False
9,mergedPRReviewCoveragePct,"SRC-002, SRC-004",14,MERGED_IN_PERIOD,PLANNED_ENRICHMENT,False



EIP | Ingestion Feasibility Validation


,validationRule,validationStatus
0,Basic historical boundary reached,PASS
1,Selected enrichment cohort reconciled,PASS
2,Enrichment identifiers unique,PASS
3,Proposed request budget respected,PASS
4,Primary API reserve preserved in estimate,PASS
5,Basic historical benchmark window sufficient,PASS



EIP | FINAL INGESTION RECOMMENDATION
Selected Strategy: 14-day merged cohort enrichment
Basic Historical Coverage: 60 days
Observed Basic PR Candidates: 4707
Detailed Enrichment Coverage: 14 days
Selected Enrichment PRs: 430
Planned Repository Requests: 1
Planned PR List Requests: 50
Planned PR Detail Requests: 430
Minimum PR Review Requests: 430
Baseline API Requests: 911
Contingency Requests: 183
Estimated Total API Requests: 1094
Effective Demo Budget: 1200
Estimated Budget Headroom: 106
All Ingestion Feasibility Checks Passed: True
Additional API Requests Executed: 0
Guaranteed Historical Coverage: False
Historical Details Extracted: False
Historical Reviews Extracted: False
Production Ingestion Implemented: False
Database Persistence Completed: False

Ingestion Planning Status: COMPLETED_WITH_LIMITATIONS


# 06. Historical Profiling Findings and Conclusions

## 6.1 Executive Summary

Completed historical GitHub source profiling for the
Engineering Intelligence Platform (EIP).

The exploration assessed historical source volumes,
data quality, lifecycle activity, reporting benchmarks,
API consumption and alternative ingestion strategies.

A selective enrichment approach was selected to support
meaningful engineering intelligence while maintaining
a bounded and efficient source extraction workload.

## 6.2 Source Extraction Results

| Attribute | Result |
|---|---|
| Source System | GitHub REST API |
| Repository | apache/airflow |
| Reporting End Exclusive | 2026-11-26 00:00 UTC |
| Maximum Evaluated Window | 90 days |
| Pages Retrieved | 50 |
| Raw Source Observations | 5000 |
| Discovered Source Attributes | 368 |
| Valid Unique Pull Requests | 4999 |
| Duplicate Valid Observations | 0 |
| Quarantined Observations | 1 |
| Observed Update Lookback | 63.17 days |
| Pages With Ordering Warnings | 29 |
| Pages With Cross-Page Overlap | 13 |
| Extraction Stop Reason | MAXIMUM_PAGES_REACHED |
| Record Count Reconciliation | PASS |

Historical source coverage is estimated rather than
guaranteed because the upstream result set is live
and contains ordering inconsistencies.

## 6.3 Historical Volume Assessment

| Window | Candidate PRs | Opened | Merged | Median Merge Hours |
|---|---:|---:|---:|---:|
| 7 Days | 773 | 305 | 208 | 66.53 |
| 14 Days | 1219 | 578 | 430 | 61.72 |
| 30 Days | 2991 | 1717 | 1305 | 19.41 |
| 60 Days | 4707 | 3449 | 2603 | 20.18 |
| 90 Days | 4999 | 4045 | 2714 | 20.25 |

The 90-day window represents incomplete observed coverage.

The 30-day and 60-day windows contain a known
quarantined pull request.

Candidate PR counts are based on the latest observed
update timestamp, while lifecycle activity counts are
based on their corresponding event timestamps.

## 6.4 Source Data Quality Findings

### Source Ordering

29 of the 50 retrieved pages contained update timestamps
that were not strictly ordered.

13 pages contained timestamp ranges overlapping the
preceding page.

Historical boundary confirmation therefore uses a
conservative two-page confirmation heuristic.

This heuristic does not guarantee exhaustive coverage.

### Invalid Source Chronology

Pull request 76202 contained the following timestamps:

| Attribute | Source Value (UTC) |
|---|---|
| Created At | 2026-11-01 09:54:42 |
| Updated At | 2026-11-01 09:01:49 |
| Closed At | 2026-11-01 09:01:47 |
| Merged At | 2026-11-01 09:01:47 |

Update, closure and merge timestamps preceded creation.

The observation was quarantined without modifying
its original source values.

No valid alternative observation was available.

### Production Implications

- Preserve original observations in Bronze.
- Record data quality failures with source lineage.
- Exclude invalid observations from analytical processing.
- Disclose affected reporting periods.
- Avoid silently correcting source timestamps.
- Reconcile historical coverage periodically.

## 6.5 Historical Engineering Activity

### Non-Overlapping Rolling Reporting Periods

| Period Start | Period End Exclusive | Opened | Merged | Median Merge Hours |
|---|---|---:|---:|---:|
| 2026-10-22 | 2026-10-29 | 519 | 404 | 6.56 |
| 2026-10-29 | 2026-11-05 | 539 | 416 | 9.62 |
| 2026-11-05 | 2026-11-12 | 396 | 302 | 43.71 |
| 2026-11-12 | 2026-11-19 | 273 | 222 | 49.67 |
| 2026-11-19 | 2026-11-26 | 305 | 208 | 66.53 |

### Latest Period Comparison

| Metric | Previous | Current | Change (%) |
|---|---:|---:|---:|
| Pull Requests Opened | 273 | 305 | 11.72 |
| Pull Requests Merged | 222 | 208 | -6.31 |
| Pull Requests Closed | 270 | 300 | 11.11 |
| Closed Without Merge | 48 | 92 | 91.67 |
| Median Merge Hours | 49.67 | 66.53 | 33.94 |

### Four-Period Historical Benchmarks

| Metric | Current | Historical Average | Change (%) |
|---|---:|---:|---:|
| PRs Opened | 305 | 431.75 | -29.36 |
| PRs Merged | 208 | 336.00 | -38.10 |
| PRs Closed | 300 | 412.75 | -27.32 |
| Median Merge Hours | 66.53 | 27.39 | 142.89 |
| P90 Merge Hours | 532.73 | 419.95 | 26.85 |

The historical benchmark excludes the current period.

The benchmark for median merge duration represents
the average of four period-level medians, rather than
the pooled median across all historical PRs.

## 6.6 Preliminary Engineering Intelligence Findings

Observed median merge duration increased across five
consecutive non-overlapping reporting periods.

The latest period recorded:

- 208 merged pull requests.
- A 6.31% decline in observed merge throughput.
- A 33.94% increase in median merge duration.
- Median merge duration 142.89% above the four-period
  historical benchmark.

Potential contributing factors for investigation include:

- Increased time to first external review.
- Increased elapsed time after the first review.
- Changes in pull request size.
- Changes in the composition of merged pull requests.

These are investigative hypotheses, not established
causal explanations.

Historical review and detail enrichment is required
before evaluating these hypotheses.

## 6.7 Ingestion Strategy Evaluation

All evaluated strategies include basic PR list ingestion
with a target historical coverage of 60 days.

| Strategy | Baseline Requests | Estimated Total Requests |
|---|---:|---:|
| STR-001: Basic Ingestion | 51 | 62 |
| STR-002: Full 60-Day Enrichment | 9465 | 11358 |
| STR-003: 7-Day Merged Enrichment | 467 | 561 |
| STR-004: 14-Day Reviews Only | 481 | 578 |
| STR-005: 14-Day Merged Enrichment | 911 | 1094 |
| STR-006: 30-Day Merged Enrichment | 2661 | 3194 |

Estimates include a 20% planning allowance for
additional pagination and retries.

Actual request requirements may vary.

## 6.8 Selected Production Ingestion Strategy

Selected Strategy: STR-005

Strategy Name: 14-Day Merged Cohort Enrichment

| Source Entity | Target Coverage | Observed Volume |
|---|---|---:|
| Repository Metadata | Current | 1 |
| Pull Requests | 60 Days | 4707 |
| Pull Request Details | 14-Day Merged Cohort | 430 |
| Pull Request Reviews | 14-Day Merged Cohort | 430 Parent PRs |

### Enrichment Cohorts

| Reporting Period | Merged PRs |
|---|---:|
| Previous Period: Nov 12-18 | 222 |
| Current Period: Nov 19-25 | 208 |
| Total | 430 |

### Estimated API Consumption

| Request Category | Requests |
|---|---:|
| Repository Metadata | 1 |
| Pull Request List | 50 |
| Pull Request Details | 430 |
| Minimum Review Requests | 430 |
| Baseline Requests | 911 |
| Contingency Requests | 183 |
| Estimated Total | 1094 |
| Maximum Run Budget | 1200 |
| Estimated Budget Headroom | 106 |

The selected strategy reduces estimated request
consumption by approximately 90.4% relative to
full 60-day enrichment.

## 6.9 Production Configuration Defaults

| Configuration | Value |
|---|---|
| basicHistoryDays | 60 |
| enrichmentHistoryDays | 14 |
| enrichmentCohort | MERGED_PULL_REQUESTS |
| reportingPeriodDays | 7 |
| benchmarkPeriodCount | 4 |
| maxListPages | 50 |
| recordsPerPage | 100 |
| maxApiRequestsPerRun | 1200 |
| minimumApiReserve | 500 |
| requestContingencyPct | 20 |
| incrementalOverlapHours | 48 |
| reportingTimezone | UTC |
| boundaryConfirmationPages | 2 |

These are proposed MVP defaults and must remain
configurable for different repository activity levels.

The production pipeline must enforce request limits
across pagination requests and retries.

Actual API quota must be reassessed before ingestion.

## 6.10 Analytical Metric Availability

### Basic Lifecycle Metrics

Target Coverage: 60 Days

| Metric ID | Status |
|---|---|
| prsOpened | Source Available |
| prsMerged | Source Available |
| prsClosed | Source Available |
| prsClosedWithoutMerge | Source Available |
| medianMergeHours | Source Available |
| p90MergeHours | Source Available |

### Detailed Collaboration Metrics

Target Coverage: 14-Day Merged Cohort

| Metric ID | Status |
|---|---|
| medianFirstReviewHours | Planned Enrichment |
| meanPreReviewHours | Planned Enrichment |
| meanPostReviewHours | Planned Enrichment |
| preReviewTimeSharePct | Planned Enrichment |
| mergedPRReviewCoveragePct | Planned Enrichment |
| uniqueExternalReviewers | Planned Enrichment |
| medianLinesChanged | Planned Enrichment |
| medianFilesChanged | Planned Enrichment |

Historical four-period benchmarks have been demonstrated
for five basic lifecycle metrics.

The closed-without-merge benchmark has not yet been
calculated but is feasible using the available records.

Detailed metrics initially support comparison across
the two most recent rolling seven-day periods only.

## 6.11 Enrichment Coverage Requirements

Selective enrichment requires explicit tracking of
source extraction completeness.

The proposed Silver PR entity must include:

- isDetailExtractionComplete
- isReviewExtractionComplete
- detailExtractedAt
- reviewExtractedAt

The proposed Gold period fact entity must include:

- eligiblePRCount
- enrichedPRCount
- sampleSize
- enrichmentCoveragePct
- coverageStatus

Missing enrichment must not be interpreted as zero
review activity or zero code changes.

A completely extracted review dataset with no qualifying
reviews is distinct from a dataset not yet extracted.

## 6.12 Proposed Incremental Ingestion Strategy

- Maintain a persisted source update watermark.
- Apply a configurable 48-hour overlapping window.
- Retrieve recently updated pull requests.
- Deduplicate observations using stable source identifiers.
- Refresh required details and reviews.
- Enrich newly eligible merged pull requests.
- Persist raw observations in physical Bronze tables.
- Upsert validated Silver entities.
- Recalculate affected Gold analytical facts.
- Record execution history and data quality results.
- Periodically reconcile historical source coverage.

Watermark advancement must depend on successful
processing and recorded source coverage.

## 6.13 Validation Summary

| Validation Category | Result |
|---|---|
| Source Observation Reconciliation | PASS |
| Historical Volume Validation | 6 of 6 Passed |
| Weekly Activity Reconciliation | 8 of 8 Passed |
| Ingestion Feasibility Validation | 6 of 6 Passed |
| Historical Benchmarks Generated | 5 |
| Historical Benchmarks Available | 5 |
| Known Quarantined Observations | 1 |

Passing internal reconciliation checks does not
establish exhaustive source completeness.

## 6.14 Known Limitations

- Historical extraction was bounded to 50 list pages.
- The 90-day historical boundary was not reached.
- Upstream ordering inconsistencies were observed.
- One source observation was quarantined.
- Historical PR state changes cannot be fully reconstructed.
- Historical reopening events have not been extracted.
- Current draft status does not reconstruct review readiness.
- Historical details and reviews remain uncollected.
- Detailed metrics are restricted to the merged PR cohort.
- Four-period benchmarks are descriptive, not calibrated
  statistical anomaly models.
- API request estimates do not guarantee execution success.
- Physical database persistence remains unimplemented.

## 6.15 Next Steps

1. Finalize source registry and metric registry definitions.
2. Finalize source-to-target column mappings.
3. Implement secure production GitHub API client.
4. Implement physical Bronze tables.
5. Implement Silver transformations and validation.
6. Implement Gold lifecycle and period facts.
7. Implement deterministic insights endpoint.
8. Implement LLM reasoning and evidence validation.
9. Implement narrative endpoint and lightweight UI.
10. Complete automated tests and submission documentation.

## 6.16 Completion Status

Initial Source Exploration: COMPLETE

Historical Data Profiling: COMPLETE WITH LIMITATIONS

Preliminary Metric Feasibility: COMPLETE

Initial Ingestion Planning: COMPLETE

Production API Client: PENDING

Database Persistence: PENDING

AI Intelligence: PENDING
